# **Jio CareOps Network Issue Analyzer | Student Starter**

## **Your Engineering Handover**

You have joined the Jio Customer Care and Network Operations team as a **Junior Python Developer**. The CareOps lead spends hours switching between customer complaint records, network incident reports, recharge failure logs, and service requests to understand which cases need attention. You have been asked to build a reproducible **customer complaint and network issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, a complaint-level investigation report, and a concise evidence-based handover to Customer Care and Network Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `jio_complaints_master.csv` · `jio_incident_histories.xlsx` · `jio_service_requests.json`  
**Data:** Fictional training records; not actual Jio customer or network operations records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1hgqvHzcqbEDG5e_sjSXjPljwV2_u7noh/view?usp=sharing)

In [77]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['complaints.csv', 'network_issue_reports.xlsx', 'careops_reference.json']
ARCHIVE = 'jio_careops_project_datasets.zip'
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES) and (DATA_DIR/ARCHIVE).is_file():
 with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
  names=z.namelist()
  if sorted(names)!=sorted(REQUIRED_FILES):
   raise ValueError(f'Dataset ZIP must contain exactly the three expected files: {REQUIRED_FILES}; found {names}')
  z.extractall(DATA_DIR)
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES):
 try:
  from google.colab import files
  print('Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).')
  files.upload()
 except ImportError:
  print('Place the dataset ZIP or three extracted files in',DATA_DIR)
 if (DATA_DIR/ARCHIVE).is_file():
  with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
   if sorted(z.namelist())!=sorted(REQUIRED_FILES):
    raise ValueError('Unexpected files in dataset archive')
   z.extractall(DATA_DIR)
missing=[name for name in REQUIRED_FILES if not (DATA_DIR/name).exists()]
if missing: raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Verified the three required Jio sources:', ', '.join(REQUIRED_FILES))

Verified the three required Jio sources: complaints.csv, network_issue_reports.xlsx, careops_reference.json


## Milestone 01 · Establish the complaint baseline

**From the team:** The CareOps lead wants a transparent complaint intake audit before any triage recommendations are trusted.

**Your responsibility:** Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.

**Handover evidence:** Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.

In [78]:
import pandas as pd
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('/content')
complaints_path = DATA_DIR / 'complaints.csv'

if not complaints_path.exists():
    raise FileNotFoundError(
        f"Complaint dataset not found: {complaints_path}"
    )

complaints_df = pd.read_csv(complaints_path)

print("Complaint dataset loaded successfully!")
print("1. DATASET SHAPE")
print(f"Total rows: {complaints_df.shape[0]}")
print(f"Total columns: {complaints_df.shape[1]}")

print("\nColumn names:")
print(complaints_df.columns.tolist())
print("2. SAMPLE COMPLAINT RECORDS")

display(complaints_df.head())

print("3. UNIQUE COMPLAINT AND CUSTOMER IDs")

required_columns = ['complaint_id', 'customer_id']

missing_columns = [
    column for column in required_columns
    if column not in complaints_df.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns missing from dataset: {missing_columns}"
    )

print(
    "Unique complaint IDs:",
    complaints_df['complaint_id'].nunique()
)

print(
    "Unique customer IDs:",
    complaints_df['customer_id'].nunique()
)

print("4. DUPLICATE RECORDS")

print(
    "Duplicate complaint ID records:",
    complaints_df['complaint_id'].duplicated().sum()
)

print(
    "Fully duplicated rows:",
    complaints_df.duplicated().sum()
)

print("5. MISSING VALUES BY COLUMN")

missing_report = pd.DataFrame({
    'missing_count': complaints_df.isna().sum(),
    'missing_percentage': (
        complaints_df.isna().mean() * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    by='missing_count',
    ascending=False
)

display(missing_report)

print("6. LEADING RAW COMPLAINT CATEGORIES")

if 'issue_category' in complaints_df.columns:
    category_counts = (
        complaints_df['issue_category']
        .value_counts(dropna=False)
        .head(10)
    )

    display(category_counts)
else:
    print("The issue_category column was not found.")

print("MILESTONE 01 COMPLETED")

print("Dataset variable: complaints_df")
print(f"Total complaint records: {len(complaints_df)}")
print(
    "Total unique complaints:",
    complaints_df['complaint_id'].nunique()
)
print(
    "Total unique customers:",
    complaints_df['customer_id'].nunique()
)


Complaint dataset loaded successfully!
1. DATASET SHAPE
Total rows: 5000
Total columns: 15

Column names:
['complaint_id', 'customer_id', 'cell_site_id', 'complaint_date', 'complaint_time', 'city', 'circle', 'complaint_channel', 'issue_category', 'issue_description', 'complaint_status', 'sentiment_tag', 'network_type', 'device_type', 'amount_impacted']
2. SAMPLE COMPLAINT RECORDS


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,NaN,SIM activation or verification is pending afte...,Open,Urgent,VoLTE,JioPhone,799.0
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,NaN,SIM activation or verification is pending afte...,NaN,Urgent,VoLTE,iOS,399.0
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,NaN,Fiber connection outage reported for apartment...,In Progress,Angry,5G,JioPhone,199.0
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,Open,Frustrated,VoLTE,iOS,399.0
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,closeD,Urgent,Fiber,JioPhone,399.0


3. UNIQUE COMPLAINT AND CUSTOMER IDs
Unique complaint IDs: 4901
Unique customer IDs: 1218
4. DUPLICATE RECORDS
Duplicate complaint ID records: 99
Fully duplicated rows: 0
5. MISSING VALUES BY COLUMN


,missing_count,missing_percentage
complaint_status,459,9.18
amount_impacted,429,8.58
issue_category,290,5.80
complaint_date,6,0.12
cell_site_id,0,0.00
customer_id,0,0.00
complaint_id,0,0.00
circle,0,0.00
city,0,0.00
complaint_time,0,0.00


6. LEADING RAW COMPLAINT CATEGORIES


,count
issue_category,
network down,416
recharge failed,356
SLOW DATA,326
No Signal,321
call drop,312
internet not working,299
NaN,290
network outage,267
service request pending,265


MILESTONE 01 COMPLETED
Dataset variable: complaints_df
Total complaint records: 5000
Total unique complaints: 4901
Total unique customers: 1218


## Milestone 02 · Onboard the XLSX network source and JSON reference collections

**From the team:** Network engineering provides Excel incident logs, while customer, site, recharge, and service-request histories are organized as distinct collections inside a single JSON reference file.

**Your responsibility:** Read the workbook and JSON; materialize five independent source DataFrames and policy notes. Do not merge repeated recharge or request records prematurely.

**Required evidence:** `customers_df`, `cell_sites_df`, `network_df`, `recharge_df`, `requests_df`, `operations_notes`, and source row counts.

In [79]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import json

DATA_DIR = Path('/content')

json_path = DATA_DIR / 'careops_reference.json'
xlsx_path = DATA_DIR / 'network_issue_reports.xlsx'

with open(json_path, 'r', encoding='utf-8') as file:
    reference_data = json.load(file)

print("1. JSON REFERENCE COLLECTIONS")

print("Available JSON collections and fields:")

for collection_name, collection_data in reference_data.items():
    if isinstance(collection_data, list):
        print(
            f"{collection_name}: "
            f"{len(collection_data)} records"
        )
    else:
        print(f"{collection_name}: text/metadata")

customers_df = pd.DataFrame(
    reference_data['customers']
)

cell_sites_df = pd.DataFrame(
    reference_data['cell_sites']
)

print("2. CUSTOMER SOURCE")

print("Customers shape:", customers_df.shape)
display(customers_df.head())

print("3. CELL SITE SOURCE")

print("Cell sites shape:", cell_sites_df.shape)
display(cell_sites_df.head())

network_df = pd.read_excel(
    xlsx_path,
    sheet_name='network_issue_reports'
)

print("4. NETWORK INCIDENT SOURCE")

print("Network reports shape:", network_df.shape)

print("\nSample network reports:")
display(network_df.head())

print("\nNetwork report identifier summary:")

print(
    "Unique network report IDs:",
    network_df['network_report_id'].nunique()
)

print(
    "Duplicate network report IDs:",
    network_df['network_report_id'].duplicated().sum()
)

print("\nNetwork incident severity counts:")

display(
    network_df['severity']
    .value_counts(dropna=False)
)

recharge_df = pd.DataFrame(
    reference_data['recharge_failures']
)

requests_df = pd.DataFrame(
    reference_data['service_requests']
)

print("5. RECHARGE FAILURE SOURCE")

print("Recharge records shape:", recharge_df.shape)
display(recharge_df.head())

print("\nRecharge status counts:")

display(
    recharge_df['recharge_status']
    .value_counts(dropna=False)
)

print("\nRefund status counts:")

display(
    recharge_df['refund_status']
    .value_counts(dropna=False)
)

print("6. SERVICE REQUEST SOURCE")

print("Service request records shape:", requests_df.shape)
display(requests_df.head())

print("\nService request status counts:")

display(
    requests_df['service_status']
    .value_counts(dropna=False)
)

print("\nService request type counts:")

display(
    requests_df['request_type']
    .value_counts(dropna=False)
)

operations_notes = str(
    reference_data['network_operations_notes']
)

print("7. NETWORK OPERATIONS NOTES")

print("Notes data type:", type(operations_notes).__name__)
print("Total characters:", len(operations_notes))

print("\nFirst 1000 characters of operational notes:")
print(operations_notes[:1000])

print("8. SOURCE RECORD COUNTS")

source_counts = pd.DataFrame({
    'source': [
        'customers_df',
        'cell_sites_df',
        'network_df',
        'recharge_df',
        'requests_df'
    ],
    'row_count': [
        len(customers_df),
        len(cell_sites_df),
        len(network_df),
        len(recharge_df),
        len(requests_df)
    ]
})

display(source_counts)

print("\nMilestone 02 source ingestion completed successfully.")


1. JSON REFERENCE COLLECTIONS
Available JSON collections and fields:
dataset_description: text/metadata
customers: 1200 records
cell_sites: 1000 records
recharge_failures: 1500 records
service_requests: 1500 records
network_operations_notes: text/metadata
2. CUSTOMER SOURCE
Customers shape: (1200, 9)


,customer_id,customer_name,customer_segment,plan_type,home_city,circle,account_age_days,preferred_service,monthly_recharge_value
0,CUST00001,Krishna Mehta,Prepaid,4G Value,Indore,Madhya Pradesh,581,Mobile Data,1499
1,CUST00002,Vivaan Sharma,Prepaid,5G Unlimited,Delhi,Delhi NCR,905,Voice,1499
2,CUST00003,Reyansh Kulkarni,Prepaid,4G Value,Lucknow,UP East,1849,Roaming,399
3,CUST00004,Sai Kulkarni,Postpaid,Fiber Silver,Mumbai,Mumbai,1148,Voice,299
4,CUST00005,Myra Gupta,Prepaid,Fiber Silver,Kolkata,West Bengal,1418,Roaming,399


3. CELL SITE SOURCE
Cell sites shape: (1000, 9)


,cell_site_id,site_name,city,circle,tower_type,site_status,avg_signal_strength_dbm,capacity_utilization_pct,last_maintenance_date
0,SITE0001,DEL-NW-0001,Delhi,Delhi NCR,Indoor Booster,Active,-73.7,77.0,2025-12-23
1,SITE0002,DEL-NW-0002,Delhi,Delhi NCR,Macro Tower,Active,-86.3,48.8,2025-06-09
2,SITE0003,LUC-NW-0003,Lucknow,UP East,Macro Tower,Under Maintenance,-107.8,42.2,2025-11-11
3,SITE0004,CHE-NW-0004,Chennai,Tamil Nadu,Small Cell,Active,-103.2,70.0,2025-10-07
4,SITE0005,MUM-NW-0005,Mumbai,Mumbai,Indoor Booster,Active,-115.5,57.9,2025-10-29


4. NETWORK INCIDENT SOURCE
Network reports shape: (1300, 10)

Sample network reports:


,network_report_id,cell_site_id,city,circle,report_date,issue_type,severity,affected_customers,avg_downtime_minutes,current_status
0,NW000001,SITE0964,Indore,Madhya Pradesh,2026-05-25,Indoor coverage issue,critical,891,517,Monitoring
1,NW000002,SITE0972,Chennai,Tamil Nadu,2026-05-15,Tower outage,Critical,1590,447,Resolved
2,NW000003,SITE0646,Surat,Gujarat,2026-02-01,Fiber cut,Low,170,252,Monitoring
3,NW000004,SITE0275,Chennai,Tamil Nadu,2026-03-08,Fiber cut,critical,197,717,pending
4,NW000005,SITE0836,Surat,Gujarat,2026-05-06,Fiber cut,critical,194,233,Open


... (13 outputs left out)

,count
request_type,
Billing Investigation,262
SIM Replacement,258
Port Verification,248
Network Visit,248
Recharge Support,247
Fiber Technician,237


7. NETWORK OPERATIONS NOTES
Notes data type: str
Total characters: 8734

First 1000 characters of operational notes:
001. Circle Ops observed repeated call drop complaints near Ahmedabad commuter corridor.
002. Manager asks developer to validate orphan complaint and service request links before final reporting.
003. NOC recommends checking high capacity utilization sites in Kolkata.
004. Circle Ops observed repeated call drop complaints near Surat commuter corridor.
005. Circle Ops observed repeated call drop complaints near Kolkata commuter corridor.
006. Recharge Ops flagged delayed refund batches for UPI payment mode in Bengaluru.
007. Circle Ops observed repeated call drop complaints near Jaipur commuter corridor.
008. Recharge Ops flagged delayed refund batches for UPI payment mode in Lucknow.
009. Circle Ops observed repeated call drop complaints near Surat commuter corridor.
010. PM reminder: final notebook must show output tables, GUI evidence, and export confirmation.
011. Man

,source,row_count
0,customers_df,1200
1,cell_sites_df,1000
2,network_df,1300
3,recharge_df,1500
4,requests_df,1500



Milestone 02 source ingestion completed successfully.


## Milestone 03 · Validate and standardize complaint records

**From the team:** Quality assurance warns that missing issue labels, mixed-case statuses, and invalid timestamps make complaint-age comparisons unreliable.

**Your responsibility:** Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.

**Handover evidence:** Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.

In [80]:
import pandas as pd
from IPython.display import display

complaints_clean = complaints_df.copy()

initial_rows = len(complaints_clean)

text_columns = complaints_clean.select_dtypes(
    include=['object', 'string']
).columns

for column in text_columns:
    complaints_clean[column] = complaints_clean[column].replace(
        r'^\s*$',
        pd.NA,
        regex=True
    )

if 'complaint_date' in complaints_clean.columns:
    complaints_clean['complaint_date'] = pd.to_datetime(
        complaints_clean['complaint_date'],
        errors='coerce'
    )

    complaints_clean['invalid_complaint_date'] = (
        complaints_clean['complaint_date'].isna()
    )
else:
    complaints_clean['invalid_complaint_date'] = True

if 'last_maintenance_date' in complaints_clean.columns:
    complaints_clean['last_maintenance_date'] = pd.to_datetime(
        complaints_clean['last_maintenance_date'],
        errors='coerce'
    )

numeric_columns = [
    'amount_impacted',
    'monthly_recharge_value',
    'account_age_days',
    'avg_signal_strength_dbm',
    'capacity_utilization_pct'
]

for column in numeric_columns:
    if column in complaints_clean.columns:
        complaints_clean[column] = pd.to_numeric(
            complaints_clean[column],
            errors='coerce'
        )

if 'issue_category' in complaints_clean.columns:

    complaints_clean['issue_category_raw'] = (
        complaints_clean['issue_category']
    )

    complaints_clean['issue_category'] = (
        complaints_clean['issue_category']
        .astype('string')
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
    )

    complaints_clean['issue_group'] = (
        complaints_clean['issue_category']
        .replace({
            'network issue': 'network',
            'network problem': 'network',
            'internet issue': 'network',
            'slow internet': 'network',
            'call drop': 'network',
            'call drops': 'network',
            'weak signal': 'network',
            'recharge issue': 'recharge',
            'recharge failure': 'recharge',
            'failed recharge': 'recharge',
            'billing issue': 'billing',
            'payment issue': 'billing',
            'customer service': 'service_request',
            'service request': 'service_request'
        })
    )

    complaints_clean['issue_group'] = (
        complaints_clean['issue_group']
        .fillna('unknown')
    )

else:
    raise KeyError("Required column 'issue_category' not found.")

if 'complaint_status' in complaints_clean.columns:
    complaints_clean['complaint_status'] = (
        complaints_clean['complaint_status']
        .astype('string')
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
    )

for column in ['city', 'circle']:
    if column in complaints_clean.columns:
        complaints_clean[column] = (
            complaints_clean[column]
            .astype('string')
            .str.strip()
            .str.title()
        )

for column in ['customer_id', 'cell_site_id']:
    if column in complaints_clean.columns:
        complaints_clean[column] = (
            complaints_clean[column]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
        )

if 'customer_id' in complaints_clean.columns:
    complaints_clean['missing_customer_reference'] = (
        complaints_clean['customer_id'].isna()
    )
else:
    complaints_clean['missing_customer_reference'] = True

if 'cell_site_id' in complaints_clean.columns:
    complaints_clean['missing_site_reference'] = (
        complaints_clean['cell_site_id'].isna()
    )
else:
    complaints_clean['missing_site_reference'] = True

if 'complaint_id' not in complaints_clean.columns:
    raise KeyError("Required column 'complaint_id' not found.")

duplicate_complaint_rows = (
    complaints_clean['complaint_id'].notna()
    & complaints_clean['complaint_id'].duplicated(keep='first')
)

duplicate_complaint_count = int(duplicate_complaint_rows.sum())

complaints_clean = pd.concat([
    complaints_clean.loc[
        complaints_clean['complaint_id'].isna()
    ],
    complaints_clean.loc[
        ~complaints_clean['complaint_id'].isna()
    ].drop_duplicates(
        subset=['complaint_id'],
        keep='first'
    )
], ignore_index=True)

quality_summary = pd.DataFrame({
    'quality_check': [
        'Initial complaint rows',
        'Rows removed due to duplicate complaint IDs',
        'Final complaint rows',
        'Unique non-missing complaint IDs',
        'Missing complaint IDs',
        'Missing customer references',
        'Missing cell-site references',
        'Invalid or missing complaint dates',
        'Missing issue categories',
        'Missing complaint statuses'
    ],
    'count': [
        initial_rows,
        duplicate_complaint_count,
        len(complaints_clean),
        complaints_clean['complaint_id'].nunique(),
        complaints_clean['complaint_id'].isna().sum(),
        complaints_clean['missing_customer_reference'].sum(),
        complaints_clean['missing_site_reference'].sum(),
        complaints_clean['invalid_complaint_date'].sum(),
        complaints_clean['issue_category'].isna().sum(),
        complaints_clean['complaint_status'].isna().sum()
    ]
})

missing_value_summary = (
    complaints_clean.isna()
    .sum()
    .rename_axis('column')
    .reset_index(name='missing_count')
)

non_missing_ids = complaints_clean.loc[
    complaints_clean['complaint_id'].notna(),
    'complaint_id'
]

assert non_missing_ids.is_unique, (
    "Duplicate non-missing complaint IDs remain."
)

print("MILESTONE 03 — CLEANED COMPLAINT DATA")

print("Original rows:", initial_rows)
print("Cleaned rows:", len(complaints_clean))
print("Removed duplicate complaint-ID rows:",
      duplicate_complaint_count)

print("\nCleaned complaint sample:")
display(complaints_clean.head())

print("\nIssue group distribution:")
display(
    complaints_clean['issue_group']
    .value_counts(dropna=False)
    .rename_axis('issue_group')
    .reset_index(name='complaint_count')
)

print("\nQuality summary:")
display(quality_summary)

print("\nMissing values by column after cleaning:")
display(missing_value_summary)

print("\nComplaint-level uniqueness verified.")
print("Required outputs: complaints_clean, quality_summary")


MILESTONE 03 — CLEANED COMPLAINT DATA
Original rows: 5000
Cleaned rows: 4901
Removed duplicate complaint-ID rows: 99

Cleaned complaint sample:


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted,invalid_complaint_date,issue_category_raw,issue_group,missing_customer_reference,missing_site_reference
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,<NA>,SIM activation or verification is pending afte...,open,Urgent,VoLTE,JioPhone,799.0,False,NaN,unknown,False,False
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,<NA>,SIM activation or verification is pending afte...,<NA>,Urgent,VoLTE,iOS,399.0,False,NaN,unknown,False,False
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,<NA>,Fiber connection outage reported for apartment...,in progress,Angry,5G,JioPhone,199.0,False,NaN,unknown,False,False
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,open,Frustrated,VoLTE,iOS,399.0,False,internet not working,internet not working,False,False
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,closed,Urgent,Fiber,JioPhone,399.0,False,payment deducted recharge not done,payment deducted recharge not done,False,False



Issue group distribution:


,issue_group,complaint_count
0,recharge failed,599
1,network,549
2,network down,414
3,slow data,323
4,no signal,313
5,internet not working,295
6,unknown,282
7,network outage,263
8,service request pending,257
9,data speed low,241



Quality summary:


,quality_check,count
0,Initial complaint rows,5000
1,Rows removed due to duplicate complaint IDs,99
2,Final complaint rows,4901
3,Unique non-missing complaint IDs,4901
4,Missing complaint IDs,0
5,Missing customer references,0
6,Missing cell-site references,0
7,Invalid or missing complaint dates,12
8,Missing issue categories,282
9,Missing complaint statuses,452



Missing values by column after cleaning:


,column,missing_count
0,complaint_id,0
1,customer_id,0
2,cell_site_id,0
3,complaint_date,12
4,complaint_time,0
5,city,0
6,circle,0
7,complaint_channel,0
8,issue_category,282
9,issue_description,0



Complaint-level uniqueness verified.
Required outputs: complaints_clean, quality_summary


## Milestone 04 · Prepare reliable network incident signals

**From the team:** Network operations wants tower status and incident severity contextualized before escalation decisions.

**Your responsibility:** Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.

**Handover evidence:** Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.

In [81]:
import pandas as pd

network_clean = network_df.copy()
recharge_clean = recharge_df.copy()
requests_clean = requests_df.copy()

network_required = [
    "network_report_id",
    "cell_site_id",
    "severity",
    "report_date"
]

missing = [
    col for col in network_required
    if col not in network_clean.columns
]

if missing:
    raise ValueError(
        f"Missing network columns: {missing}"
    )

for col in ["network_report_id", "cell_site_id"]:
    network_clean[col] = (
        network_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

network_clean["severity"] = (
    network_clean["severity"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace("", pd.NA)
)

network_clean["severity"] = network_clean["severity"].replace({
    "crit": "critical",
    "major": "high",
    "moderate": "medium",
    "minor": "low"
})

network_clean["report_date"] = pd.to_datetime(
    network_clean["report_date"],
    errors="coerce"
)

if "current_status" in network_clean.columns:
    network_clean["current_status"] = (
        network_clean["current_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace("", pd.NA)
    )

for col in ["affected_customers", "avg_downtime_minutes"]:
    if col in network_clean.columns:
        network_clean[col] = pd.to_numeric(
            network_clean[col],
            errors="coerce"
        )

network_clean = network_clean[
    network_clean["network_report_id"].notna()
].drop_duplicates(
    subset=["network_report_id"],
    keep="first"
).reset_index(drop=True)

recharge_required = [
    "recharge_id",
    "complaint_id",
    "recharge_status",
    "recharge_date"
]

missing = [
    col for col in recharge_required
    if col not in recharge_clean.columns
]

if missing:
    raise ValueError(
        f"Missing recharge columns: {missing}"
    )

for col in ["recharge_id", "complaint_id"]:
    recharge_clean[col] = (
        recharge_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

recharge_clean["recharge_status"] = (
    recharge_clean["recharge_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace("", pd.NA)
)

recharge_clean["recharge_date"] = pd.to_datetime(
    recharge_clean["recharge_date"],
    errors="coerce"
)

recharge_clean = recharge_clean[
    recharge_clean["recharge_id"].notna()
].drop_duplicates(
    subset=["recharge_id"],
    keep="first"
).reset_index(drop=True)

requests_required = [
    "service_request_id",
    "complaint_id",
    "service_status",
    "request_created_date"
]

missing = [
    col for col in requests_required
    if col not in requests_clean.columns
]

if missing:
    raise ValueError(
        f"Missing service request columns: {missing}"
    )

for col in ["service_request_id", "complaint_id"]:
    requests_clean[col] = (
        requests_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

requests_clean["service_status"] = (
    requests_clean["service_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace("", pd.NA)
)

requests_clean["request_created_date"] = pd.to_datetime(
    requests_clean["request_created_date"],
    errors="coerce"
)

requests_clean = requests_clean[
    requests_clean["service_request_id"].notna()
].drop_duplicates(
    subset=["service_request_id"],
    keep="first"
).reset_index(drop=True)

print("MILESTONE 04 — OPERATIONAL CLEANING RESULTS")


print("\nNetwork clean shape:", network_clean.shape)
print("Recharge clean shape:", recharge_clean.shape)
print("Requests clean shape:", requests_clean.shape)

print("\nUnique network report IDs:",
      network_clean["network_report_id"].is_unique)

print("Unique recharge IDs:",
      recharge_clean["recharge_id"].is_unique)

print("Unique service request IDs:",
      requests_clean["service_request_id"].is_unique)

print("\nNetwork report date dtype:",
      network_clean["report_date"].dtype)

print("Recharge date dtype:",
      recharge_clean["recharge_date"].dtype)

print("Service request date dtype:",
      requests_clean["request_created_date"].dtype)

display(network_clean.head())
display(recharge_clean.head())
display(requests_clean.head())

assert network_clean["network_report_id"].notna().all()
assert network_clean["network_report_id"].is_unique

assert recharge_clean["recharge_id"].notna().all()
assert recharge_clean["recharge_id"].is_unique

assert requests_clean["service_request_id"].notna().all()
assert requests_clean["service_request_id"].is_unique

assert pd.api.types.is_datetime64_any_dtype(
    network_clean["report_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    recharge_clean["recharge_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    requests_clean["request_created_date"]
)

print("\nMilestone 04 validation completed.")

MILESTONE 04 — OPERATIONAL CLEANING RESULTS

Network clean shape: (1300, 10)
Recharge clean shape: (1500, 9)
Requests clean shape: (1500, 10)

Unique network report IDs: True
Unique recharge IDs: True
Unique service request IDs: True

Network report date dtype: datetime64[ns]
Recharge date dtype: datetime64[ns]
Service request date dtype: datetime64[ns]


,network_report_id,cell_site_id,city,circle,report_date,issue_type,severity,affected_customers,avg_downtime_minutes,current_status
0,NW000001,SITE0964,Indore,Madhya Pradesh,2026-05-25,Indoor coverage issue,critical,891,517,monitoring
1,NW000002,SITE0972,Chennai,Tamil Nadu,2026-05-15,Tower outage,critical,1590,447,resolved
2,NW000003,SITE0646,Surat,Gujarat,2026-02-01,Fiber cut,low,170,252,monitoring
3,NW000004,SITE0275,Chennai,Tamil Nadu,2026-03-08,Fiber cut,critical,197,717,pending
4,NW000005,SITE0836,Surat,Gujarat,2026-05-06,Fiber cut,critical,194,233,open


,recharge_id,complaint_id,customer_id,recharge_date,recharge_amount,payment_mode,recharge_status,refund_status,refund_completed_date
0,RCH000001,CMP002599,CUST00947,2026-02-07,None,upi,failed payment,pending refund,None
1,RCH000002,CMP001502,CUST00264,2026-05-29,None,upi,recharge pending,Not Initiated,None
2,RCH000003,CMP004726,CUST00498,2026-03-30,None,Cash,failed,Failed,None
3,RCH000004,CMP002853,CUST00920,2026-04-27,999,UPI,<NA>,Failed,None
4,RCH000005,CMP003419,CUST00634,2026-03-22,149,UPI,failed,Completed,2026-03-22


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag
0,SR000001,CMP004694,CUST00217,2026-04-06 02:00:00,,Recharge Support,escalated,Recharge Ops,,Y
1,SR000002,CMP001907,CUST00813,2026-03-15 00:00:00,,Recharge Support,open,Fiber Field Team,56,yes
2,SR000003,CMP003243,CUST00377,2026-05-07 21:00:00,2026-05-08 08:00,Billing Investigation,closed,Care Escalation Desk,,N
3,SR000004,CMP002745,CUST01194,2026-05-13 23:00:00,2026-05-14 08:00,Fiber Technician,closed,Retail Ops,,No
4,SR000005,CMP002664,CUST00471,2026-04-10 23:00:00,2026-04-13 06:00,Network Visit,resolved,Retail Ops,None,N



Milestone 04 validation completed.


## Milestone 05 · Audit recharge failures and refund risk

**From the team:** Recharge Support needs to spot unresolved money-impacting cases and failed payments linked to complaints.

**Your responsibility:** Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.

**Handover evidence:** Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.

In [82]:
import pandas as pd
import numpy as np

print("MILESTONE 5: NETWORK ISSUE ANALYSIS BY CELL SITE")

if "network_clean" not in globals():
    raise NameError(
        "network_clean not found. Please run Milestone 4 first."
    )

if "cell_sites_df" not in globals():
    raise NameError(
        "cell_sites_df not found. Please run Milestone 2 first."
    )

network_clean = network_clean.copy()
sites_clean = cell_sites_df.copy()

required_network_cols = [
    "network_report_id",
    "cell_site_id",
    "severity"
]

missing_cols = [
    col for col in required_network_cols
    if col not in network_clean.columns
]

if missing_cols:
    raise KeyError(
        f"Missing columns in network_clean: {missing_cols}"
    )

if "cell_site_id" not in sites_clean.columns:
    raise KeyError(
        "cell_site_id column not found in cell_sites_df."
    )

network_clean["cell_site_id"] = (
    network_clean["cell_site_id"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

sites_clean["cell_site_id"] = (
    sites_clean["cell_site_id"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

network_clean["severity"] = (
    network_clean["severity"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[_-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

print("\nStep 2 completed: Cell site IDs and severity standardized.")

severe_categories = {
    "critical",
    "high",
    "major outage"
}

network_clean["severe_network_flag"] = (
    network_clean["severity"]
    .isin(severe_categories)
    .astype("int64")
)

print("\nSeverity distribution:")
display(
    network_clean["severity"]
    .value_counts(dropna=False)
    .rename_axis("severity")
    .reset_index(name="report_count")
)

network_with_site = network_clean.loc[
    network_clean["cell_site_id"].notna()
].copy()

network_by_site = (
    network_with_site
    .groupby("cell_site_id", as_index=False)
    .agg(
        network_report_count=(
            "network_report_id",
            "count"
        ),
        severe_network_report_count=(
            "severe_network_flag",
            "sum"
        )
    )
)

network_by_site["network_report_count"] = (
    network_by_site["network_report_count"]
    .astype("int64")
)

network_by_site["severe_network_report_count"] = (
    network_by_site["severe_network_report_count"]
    .astype("int64")
)

print("\nStep 4 completed: Network reports aggregated by cell site.")

expected_total = int(
    network_clean["cell_site_id"].notna().sum()
)

actual_total = int(
    network_by_site["network_report_count"].sum()
)

assert network_by_site["cell_site_id"].is_unique, (
    "ERROR: Duplicate cell_site_id values found in network_by_site."
)

assert network_by_site["cell_site_id"].notna().all(), (
    "ERROR: Missing cell_site_id values found in network_by_site."
)

assert actual_total == expected_total, (
    f"ERROR: Network report totals do not match. "
    f"Expected {expected_total}, found {actual_total}."
)

assert (
    network_by_site["severe_network_report_count"]
    <= network_by_site["network_report_count"]
).all(), "ERROR: Severe report count exceeds total report count."

print("\nNetwork aggregation validation: PASSED")
print("Network reports with a valid cell site:", expected_total)
print("Network reports included in aggregation:", actual_total)
print("Unique cell sites:", network_by_site["cell_site_id"].nunique())
print(
    "Severe network reports:",
    int(network_by_site["severe_network_report_count"].sum())
)

sites_clean = sites_clean.dropna(
    subset=["cell_site_id"]
).drop_duplicates(
    subset=["cell_site_id"],
    keep="first"
).copy()

site_network_inventory = sites_clean.merge(
    network_by_site,
    on="cell_site_id",
    how="left",
    validate="one_to_one"
)

site_network_inventory["network_report_count"] = (
    site_network_inventory["network_report_count"]
    .fillna(0)
    .astype("int64")
)

site_network_inventory["severe_network_report_count"] = (
    site_network_inventory["severe_network_report_count"]
    .fillna(0)
    .astype("int64")
)

print("\nSite inventory with network summary:")
display(site_network_inventory.head(10))

top_affected_sites = (
    network_by_site
    .sort_values(
        by=[
            "severe_network_report_count",
            "network_report_count"
        ],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

print("\nTop 10 cell sites by severe network reports:")
display(top_affected_sites)

required_output_cols = [
    "cell_site_id",
    "network_report_count",
    "severe_network_report_count"
]

assert all(
    col in network_by_site.columns
    for col in required_output_cols
), "ERROR: Required assessment columns are missing."

assert network_by_site["cell_site_id"].is_unique
assert network_by_site["cell_site_id"].notna().all()
assert (
    network_by_site["network_report_count"].sum()
    == network_clean["cell_site_id"].notna().sum()
)

print("MILESTONE 5 COMPLETED SUCCESSFULLY")

print("\nRequired output variable: network_by_site")
print("Columns:", network_by_site.columns.tolist())

display(network_by_site.head(10))


MILESTONE 5: NETWORK ISSUE ANALYSIS BY CELL SITE

Step 2 completed: Cell site IDs and severity standardized.

Severity distribution:


,severity,report_count
0,high,381
1,critical,371
2,low,201
3,medium,174
4,major outage,173



Step 4 completed: Network reports aggregated by cell site.

Network aggregation validation: PASSED
Network reports with a valid cell site: 1300
Network reports included in aggregation: 1300
Unique cell sites: 721
Severe network reports: 925

Site inventory with network summary:


,cell_site_id,site_name,city,circle,tower_type,site_status,avg_signal_strength_dbm,capacity_utilization_pct,last_maintenance_date,network_report_count,severe_network_report_count
0,SITE0001,DEL-NW-0001,Delhi,Delhi NCR,Indoor Booster,Active,-73.7,77.0,2025-12-23,0,0
1,SITE0002,DEL-NW-0002,Delhi,Delhi NCR,Macro Tower,Active,-86.3,48.8,2025-06-09,0,0
2,SITE0003,LUC-NW-0003,Lucknow,UP East,Macro Tower,Under Maintenance,-107.8,42.2,2025-11-11,0,0
3,SITE0004,CHE-NW-0004,Chennai,Tamil Nadu,Small Cell,Active,-103.2,70.0,2025-10-07,0,0
4,SITE0005,MUM-NW-0005,Mumbai,Mumbai,Indoor Booster,Active,-115.5,57.9,2025-10-29,1,1
5,SITE0006,PUN-NW-0006,Pune,Maharashtra,Macro Tower,Active,-110.3,63.5,2026-04-22,0,0
6,SITE0007,PUN-NW-0007,Pune,Maharashtra,Small Cell,Active,-77.7,56.9,2025-06-21,0,0
7,SITE0008,AHM-NW-0008,Ahmedabad,Gujarat,Indoor Booster,Active,-90.7,56.9,2026-02-13,2,1
8,SITE0009,JAI-NW-0009,Jaipur,Rajasthan,Fiber Hub,Planned Upgrade,-103.9,38.5,2026-03-15,0,0
9,SITE0010,AHM-NW-0010,Ahmedabad,Gujarat,Fiber Hub,Active,-89.4,53.1,2025-12-10,2,1



Top 10 cell sites by severe network reports:


,cell_site_id,network_report_count,severe_network_report_count
0,SITE0136,6,4
1,SITE0375,6,4
2,SITE0164,4,4
3,SITE0228,4,4
4,SITE0386,4,4
5,SITE0424,4,4
6,SITE0481,4,4
7,SITE0524,4,4
8,SITE0537,4,4
9,SITE0648,4,4


MILESTONE 5 COMPLETED SUCCESSFULLY

Required output variable: network_by_site
Columns: ['cell_site_id', 'network_report_count', 'severe_network_report_count']


,cell_site_id,network_report_count,severe_network_report_count
0,SITE0005,1,1
1,SITE0008,2,1
2,SITE0010,2,1
3,SITE0011,2,2
4,SITE0013,3,1
5,SITE0014,1,0
6,SITE0015,4,2
7,SITE0017,1,0
8,SITE0018,1,1
9,SITE0019,2,0


## Milestone 06 · Detect service request and SLA follow-ups

**From the team:** The Service Desk wants unresolved or escalated requests visible on the original complaint without introducing duplicate complaint rows.

**Your responsibility:** Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.

**Handover evidence:** Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.

In [83]:
import pandas as pd
import numpy as np

print("MILESTONE 06: RECHARGE AND SERVICE REQUEST ANALYSIS")

if "recharge_clean" not in globals():
    raise NameError(
        "recharge_clean missing. Run the recharge cleaning cell first."
    )

recharge_clean = recharge_clean.copy()

for col in ["recharge_id", "complaint_id"]:
    if col not in recharge_clean.columns:
        raise KeyError(f"Required recharge column missing: {col}")

    recharge_clean[col] = (
        recharge_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

if "recharge_status" not in recharge_clean.columns:
    raise KeyError("recharge_status missing from recharge_clean.")

recharge_clean["recharge_status"] = (
    recharge_clean["recharge_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[_-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

recharge_status_mapping = {
    "failure": "failed",
    "fail": "failed",
    "payment failed": "failed",
    "failed payment": "failed",
    "success": "successful",
    "successfully completed": "successful",
    "completed": "successful",
    "in progress": "pending",
    "processing": "pending",
    "refund pending": "pending refund",
    "refund in progress": "pending refund"
}

recharge_clean["recharge_status"] = (
    recharge_clean["recharge_status"]
    .replace(recharge_status_mapping)
)

recharge_status = recharge_clean["recharge_status"]

recharge_clean["recharge_failed_flag"] = (
    recharge_status.eq("failed")
)

recharge_clean["recharge_pending_flag"] = (
    recharge_status.eq("pending")
)

recharge_clean["pending_refund_flag"] = (
    recharge_status.isin([
        "pending refund",
        "refund processing",
        "refund initiated"
    ])
)

recharge_clean["recharge_risk_flag"] = (
    recharge_clean["recharge_failed_flag"]
    | recharge_clean["recharge_pending_flag"]
    | recharge_clean["pending_refund_flag"]
)

linked_recharges = recharge_clean.loc[
    recharge_clean["complaint_id"].notna()
].copy()

recharge_by_complaint = (
    linked_recharges
    .groupby("complaint_id", as_index=False)
    .agg(
        recharge_count=("recharge_id", "count"),
        failed_recharge_count=("recharge_failed_flag", "sum"),
        pending_recharge_count=("recharge_pending_flag", "sum"),
        pending_refund_count=("pending_refund_flag", "sum"),
        recharge_risk_count=("recharge_risk_flag", "sum")
    )
)

recharge_count_cols = [
    "recharge_count",
    "failed_recharge_count",
    "pending_recharge_count",
    "pending_refund_count",
    "recharge_risk_count"
]

recharge_by_complaint[recharge_count_cols] = (
    recharge_by_complaint[recharge_count_cols].astype("int64")
)

assert recharge_by_complaint["complaint_id"].notna().all(), (
    "Recharge summary contains missing complaint IDs."
)

assert recharge_by_complaint["complaint_id"].is_unique, (
    "Recharge summary contains duplicate complaint IDs."
)

expected_recharge_count = int(
    linked_recharges["recharge_id"].notna().sum()
)

actual_recharge_count = int(
    recharge_by_complaint["recharge_count"].sum()
)

assert actual_recharge_count == expected_recharge_count, (
    f"Recharge counts do not reconcile: expected "
    f"{expected_recharge_count}, got {actual_recharge_count}."
)

print("\nRECHARGE SUMMARY")
print("Unique complaints:", recharge_by_complaint["complaint_id"].nunique())
print("Recharge events:", actual_recharge_count)
print(
    "Failed recharge events:",
    int(recharge_by_complaint["failed_recharge_count"].sum())
)
print(
    "Pending refund events:",
    int(recharge_by_complaint["pending_refund_count"].sum())
)
print(
    "Recharge risk events:",
    int(recharge_by_complaint["recharge_risk_count"].sum())
)

display(recharge_by_complaint.head(10))

if "requests_df" not in globals():
    raise NameError("requests_df missing. Run Milestone 2 first.")

if "complaints_clean" not in globals():
    raise NameError("complaints_clean missing. Run Milestone 3 first.")

requests_clean = requests_df.copy()
complaints_clean = complaints_clean.copy()

required_request_columns = [
    "service_request_id",
    "complaint_id",
    "customer_id",
    "request_created_date",
    "request_closed_date",
    "service_status",
    "resolution_time_hours",
    "escalation_flag"
]

missing_request_columns = [
    col for col in required_request_columns
    if col not in requests_clean.columns
]

if missing_request_columns:
    raise KeyError(
        f"Required service request columns missing: "
        f"{missing_request_columns}"
    )

for col in [
    "service_request_id",
    "complaint_id",
    "customer_id"
]:
    requests_clean[col] = (
        requests_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

requests_clean = (
    requests_clean
    .dropna(subset=["service_request_id"])
    .drop_duplicates(subset=["service_request_id"], keep="first")
    .copy()
)

assert requests_clean["service_request_id"].is_unique
assert requests_clean["service_request_id"].notna().all()

requests_clean["service_status"] = (
    requests_clean["service_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[_-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

request_status_mapping = {
    "new": "open",
    "opened": "open",
    "inprogress": "in progress",
    "work in progress": "in progress",
    "waiting": "pending",
    "awaiting action": "pending",
    "escalation": "escalated",
    "complete": "closed",
    "completed": "closed",
    "cancelled": "closed",
    "canceled": "closed"
}

requests_clean["service_status"] = (
    requests_clean["service_status"].replace(request_status_mapping)
)

requests_clean["request_created_date"] = pd.to_datetime(
    requests_clean["request_created_date"],
    errors="coerce"
)

requests_clean["request_closed_date"] = pd.to_datetime(
    requests_clean["request_closed_date"],
    errors="coerce"
)

requests_clean["resolution_time_hours"] = pd.to_numeric(
    requests_clean["resolution_time_hours"],
    errors="coerce"
)

requests_clean.loc[
    requests_clean["resolution_time_hours"] < 0,
    "resolution_time_hours"
] = np.nan

escalation_values = (
    requests_clean["escalation_flag"]
    .astype("string")
    .str.strip()
    .str.lower()
)

requests_clean["escalation_flag"] = escalation_values.isin(
    ["true", "yes", "1", "y", "escalated"]
)

closed_statuses = {"closed", "resolved"}

requests_clean["closed_request_flag"] = (
    requests_clean["service_status"].isin(closed_statuses)
)

requests_clean["unresolved_request_flag"] = (
    ~requests_clean["closed_request_flag"]
    & requests_clean["service_status"].notna()
)

requests_clean["escalated_request_flag"] = (
    requests_clean["escalation_flag"]
    | requests_clean["service_status"].eq("escalated")
)

requests_clean["sla_review_flag"] = (
    requests_clean["unresolved_request_flag"]
    & requests_clean["request_created_date"].notna()
)

print("\nSERVICE REQUEST CLEANING")

print("Unique service requests:", len(requests_clean))

print(
    "Unresolved requests:",
    int(requests_clean["unresolved_request_flag"].sum())
)
print(
    "Escalated requests:",
    int(requests_clean["escalated_request_flag"].sum())
)
print(
    "Closed/resolved requests:",
    int(requests_clean["closed_request_flag"].sum())
)
print(
    "SLA/follow-up review candidates:",
    int(requests_clean["sla_review_flag"].sum())
)

complaints_clean["complaint_id"] = (
    complaints_clean["complaint_id"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

valid_complaint_ids = set(
    complaints_clean["complaint_id"].dropna()
)

request_complaint_ids = (
    requests_clean["complaint_id"]
    .astype("string")
    .str.strip()
)

orphan_mask = (
    request_complaint_ids.isna()
    | ~request_complaint_ids.isin(valid_complaint_ids)
)

orphan_requests = requests_clean.loc[orphan_mask].copy()

print("\nORPHAN REQUEST REFERENCES")

print("Orphan requests:", len(orphan_requests))

if not orphan_requests.empty:
    display(
        orphan_requests[
            [
                "service_request_id",
                "complaint_id",
                "service_status",
                "request_created_date"
            ]
        ].head(20)
    )

linked_requests = requests_clean.loc[
    request_complaint_ids.notna()
    & request_complaint_ids.isin(valid_complaint_ids)
].copy()

requests_by_complaint = (
    linked_requests
    .groupby("complaint_id", as_index=False)
    .agg(
        request_count=("service_request_id", "count"),
        open_request_count=("unresolved_request_flag", "sum"),
        escalated_request_count=("escalated_request_flag", "sum"),
        sla_review_count=("sla_review_flag", "sum")
    )
)

request_count_cols = [
    "request_count",
    "open_request_count",
    "escalated_request_count",
    "sla_review_count"
]

requests_by_complaint[request_count_cols] = (
    requests_by_complaint[request_count_cols].astype("int64")
)

assert requests_by_complaint["complaint_id"].notna().all()
assert requests_by_complaint["complaint_id"].is_unique

assert (
    requests_by_complaint["request_count"].sum()
    == len(linked_requests)
), "Service request totals do not reconcile."

print("\nREQUEST SUMMARY BY COMPLAINT")

print("Complaints with linked requests:", len(requests_by_complaint))

print("Linked service requests:", len(linked_requests))

display(requests_by_complaint.head(10))

customer_complaints = complaints_clean.loc[
    complaints_clean["customer_id"].notna()
].copy()

customer_repeat_count = (
    customer_complaints
    .groupby("customer_id", as_index=False)
    .agg(
        customer_complaint_count=("complaint_id", "count")
    )
)

customer_repeat_count["customer_complaint_count"] = (
    customer_repeat_count["customer_complaint_count"].astype("int64")
)

assert customer_repeat_count["customer_id"].notna().all()
assert customer_repeat_count["customer_id"].is_unique

assert (
    customer_repeat_count["customer_complaint_count"].sum()
    == complaints_clean["customer_id"].notna().sum()
), "Repeat customer counts do not reconcile."

print("\nREPEAT CUSTOMER SUMMARY")

print("Unique customers:", len(customer_repeat_count))

print(
    "Complaints linked to customers:",
    int(customer_repeat_count["customer_complaint_count"].sum())
)

display(
    customer_repeat_count
    .sort_values("customer_complaint_count", ascending=False)
    .head(10)
)

print("MILESTONE 06 VALIDATION")


print("Recharge summary unique complaint IDs:",
      recharge_by_complaint["complaint_id"].is_unique)

print("Service request summary unique complaint IDs:",
      requests_by_complaint["complaint_id"].is_unique)

print("Repeat customer summary unique customer IDs:",
      customer_repeat_count["customer_id"].is_unique)

print("Orphan service requests:", len(orphan_requests))

print("\nVariables created:")
print("- recharge_clean")
print("- recharge_by_complaint")
print("- requests_clean")
print("- requests_by_complaint")
print("- orphan_requests")
print("- customer_repeat_count")

print("\nMilestone 06 processing completed successfully.")


MILESTONE 06: RECHARGE AND SERVICE REQUEST ANALYSIS

RECHARGE SUMMARY
Unique complaints: 1257
Recharge events: 1500
Failed recharge events: 641
Pending refund events: 0
Recharge risk events: 641


,complaint_id,recharge_count,failed_recharge_count,pending_recharge_count,pending_refund_count,recharge_risk_count
0,CMP000001,1,1,0,0,1
1,CMP000003,1,0,0,0,0
2,CMP000007,1,0,0,0,0
3,CMP000009,1,0,0,0,0
4,CMP000010,1,0,0,0,0
5,CMP000015,1,0,0,0,0
6,CMP000020,1,1,0,0,1
7,CMP000021,1,1,0,0,1
8,CMP000022,1,1,0,0,1
9,CMP000023,1,1,0,0,1



SERVICE REQUEST CLEANING
Unique service requests: 1500
Unresolved requests: 1089
Escalated requests: 840
Closed/resolved requests: 411
SLA/follow-up review candidates: 1089

ORPHAN REQUEST REFERENCES
Orphan requests: 3


,service_request_id,complaint_id,service_status,request_created_date
325,SR000326,CMPX943667,pending,2026-05-10 10:00:00
1138,SR001139,CMPX905803,,2026-03-31 15:00:00
1492,SR001493,CMPX928838,open,2026-03-20 05:00:00



REQUEST SUMMARY BY COMPLAINT
Complaints with linked requests: 1262
Linked service requests: 1497


,complaint_id,request_count,open_request_count,escalated_request_count,sla_review_count
0,CMP000002,1,0,1,0
1,CMP000003,1,0,0,0
2,CMP000008,1,1,0,1
3,CMP000009,2,2,1,2
4,CMP000011,1,1,0,1
5,CMP000013,2,1,1,1
6,CMP000015,2,2,2,2
7,CMP000032,1,1,1,1
8,CMP000034,1,0,0,0
9,CMP000035,1,0,1,0



REPEAT CUSTOMER SUMMARY
Unique customers: 1217
Complaints linked to customers: 4901


,customer_id,customer_complaint_count
1153,CUST01170,11
587,CUST00597,11
604,CUST00615,11
4,CUST00005,10
1114,CUST01131,10
166,CUST00169,10
519,CUST00527,10
496,CUST00504,10
95,CUST00097,10
1118,CUST01135,10


MILESTONE 06 VALIDATION
Recharge summary unique complaint IDs: True
Service request summary unique complaint IDs: True
Repeat customer summary unique customer IDs: True
Orphan service requests: 3

Variables created:
- recharge_clean
- recharge_by_complaint
- requests_clean
- requests_by_complaint
- orphan_requests
- customer_repeat_count

Milestone 06 processing completed successfully.


## Milestone 07 · Analyze city, circle and repeated customer impact

**From the team:** Circle managers need geographic clusters and repeat contacts—not misleading counts created by multi-table joins.

**Your responsibility:** Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.

**Handover evidence:** Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.

In [84]:
import pandas as pd
import numpy as np

required_dataframes = [
    "complaints_clean",
    "requests_clean",
    "cell_sites_df"
]

for dataframe_name in required_dataframes:
    if dataframe_name not in globals():
        raise NameError(
            f"DataFrame '{dataframe_name}' not found. "
            "Run the previous milestone cells first."
        )

complaints = complaints_clean.copy()
requests = requests_clean.copy()
sites = cell_sites_df.copy()

complaints.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in complaints.columns
]

requests.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in requests.columns
]

sites.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in sites.columns
]

def find_column(df, possible_names):
    for name in possible_names:
        normalized_name = name.strip().lower().replace(" ", "_")
        if normalized_name in df.columns:
            return normalized_name
    return None

if "complaint_id" not in complaints.columns:
    raise ValueError(
        "complaint_id is missing from complaints_clean. "
        f"Available columns: {complaints.columns.tolist()}"
    )

required_request_columns = [
    "service_request_id",
    "complaint_id"
]

for col in required_request_columns:
    if col not in requests.columns:
        raise ValueError(
            f"'{col}' is missing from requests_clean. "
            f"Available columns: {requests.columns.tolist()}"
        )

complaints["complaint_id"] = (
    complaints["complaint_id"].astype("string").str.strip()
)

requests["complaint_id"] = (
    requests["complaint_id"].astype("string").str.strip()
)

if "customer_id" in complaints.columns:
    complaints["customer_id"] = (
        complaints["customer_id"].astype("string").str.strip()
    )

if "customer_id" in requests.columns:
    requests["customer_id"] = (
        requests["customer_id"].astype("string").str.strip()
    )

if "service_status" in requests.columns:
    requests["_status_normalized"] = (
        requests["service_status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
else:
    requests["_status_normalized"] = ""

open_statuses = [
    "open",
    "pending",
    "in progress",
    "in_progress",
    "escalated"
]

requests["_is_open"] = (
    requests["_status_normalized"].isin(open_statuses)
    | requests["_status_normalized"].eq("")
).astype(int)

if "escalation_flag" in requests.columns:
    escalation_text = (
        requests["escalation_flag"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    escalation_flag = escalation_text.isin(
        ["1", "true", "yes", "y", "escalated"]
    )
else:
    escalation_flag = pd.Series(
        False, index=requests.index
    )

requests["_is_escalated"] = (
    requests["_status_normalized"].eq("escalated")
    | escalation_flag
).astype(int)

sla_col = find_column(
    requests,
    [
        "sla_review_flag",
        "sla_breach_flag",
        "sla_review_required",
        "sla_breached"
    ]
)

if sla_col:
    sla_text = (
        requests[sla_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    requests["_sla_review"] = sla_text.isin(
        ["1", "true", "yes", "y", "breached", "breach"]
    ).astype(int)
else:

    requests["_sla_review"] = 0

valid_request_links = requests[
    requests["complaint_id"].notna()
    & requests["complaint_id"].ne("")
].copy()

requests_by_complaint = (
    valid_request_links
    .groupby("complaint_id", as_index=False)
    .agg(
        request_count=("service_request_id", "size"),
        open_request_count=("_is_open", "sum"),
        escalated_request_count=("_is_escalated", "sum"),
        sla_review_count=("_sla_review", "sum")
    )
)

expected_request_count = len(valid_request_links)
actual_request_count = int(requests_by_complaint["request_count"].sum())

assert actual_request_count == expected_request_count, (
    f"Request counts do not reconcile: "
    f"expected {expected_request_count}, got {actual_request_count}"
)

assert requests_by_complaint["complaint_id"].is_unique, (
    "requests_by_complaint contains duplicate complaint IDs."
)

print("SERVICE REQUEST SUMMARY")
print("Requests with complaint references:", expected_request_count)
print("Grouped request total:", actual_request_count)
display(requests_by_complaint.head(10))

if "customer_id" not in complaints.columns:
    customer_repeat_count = pd.DataFrame(
        columns=["customer_id", "customer_complaint_count"]
    )
    repeat_customers = customer_repeat_count.copy()

else:
    valid_customer_complaints = complaints[
        complaints["customer_id"].notna()
        & complaints["customer_id"].ne("")
        & complaints["complaint_id"].notna()
        & complaints["complaint_id"].ne("")
    ].copy()

    customer_repeat_count = (
        valid_customer_complaints
        .groupby("customer_id", as_index=False)
        .agg(
            customer_complaint_count=("complaint_id", "nunique")
        )
    )

    repeat_customers = customer_repeat_count[
        customer_repeat_count["customer_complaint_count"] > 1
    ].copy()

    expected_customer_complaints = (
        valid_customer_complaints["complaint_id"].nunique()
    )

    actual_customer_complaints = int(
        customer_repeat_count["customer_complaint_count"].sum()
    )

    assert actual_customer_complaints == expected_customer_complaints, (
        "Customer complaint counts do not reconcile."
    )

    assert customer_repeat_count["customer_id"].is_unique, (
        "customer_repeat_count contains duplicate customer IDs."
    )

print("\nREPEAT CUSTOMER SUMMARY")
print("Unique customers:", len(customer_repeat_count))
print("Repeat customers:", len(repeat_customers))
display(customer_repeat_count.head(10))

site_id_col = find_column(
    sites,
    [
        "cell_site_id",
        "site_id",
        "cell_id",
        "site_code"
    ]
)

complaint_site_col = find_column(
    complaints,
    [
        "cell_site_id",
        "site_id",
        "cell_id",
        "site_code"
    ]
)

city_col = find_column(
    sites,
    [
        "city",
        "city_name",
        "location",
        "location_name",
        "town"
    ]
)

circle_col = find_column(
    sites,
    [
        "circle",
        "circle_name",
        "region",
        "region_name",
        "state",
        "state_name"
    ]
)

location_facts = complaints.copy()

location_facts["city"] = "Unknown"
location_facts["circle"] = "Unknown"

if site_id_col and complaint_site_col:

    site_lookup_columns = [site_id_col]

    if city_col:
        site_lookup_columns.append(city_col)

    if circle_col and circle_col not in site_lookup_columns:
        site_lookup_columns.append(circle_col)

    sites_lookup = sites[site_lookup_columns].copy()

    sites_lookup = sites_lookup.rename(
        columns={site_id_col: "cell_site_id"}
    )

    rename_location_columns = {}

    if city_col:
        rename_location_columns[city_col] = "city"

    if circle_col:
        rename_location_columns[circle_col] = "circle"

    sites_lookup = sites_lookup.rename(
        columns=rename_location_columns
    )

    sites_lookup["cell_site_id"] = (
        sites_lookup["cell_site_id"]
        .astype("string")
        .str.strip()
    )

    sites_lookup = sites_lookup.drop_duplicates(
        subset=["cell_site_id"]
    )

    if complaint_site_col != "cell_site_id":
        location_facts = location_facts.rename(
            columns={complaint_site_col: "cell_site_id"}
        )

    location_facts["cell_site_id"] = (
        location_facts["cell_site_id"]
        .astype("string")
        .str.strip()
    )

    location_facts = location_facts.drop(
        columns=["city", "circle"],
        errors="ignore"
    )

    location_facts = location_facts.merge(
        sites_lookup,
        on="cell_site_id",
        how="left",
        validate="many_to_one"
    )

    if "city" not in location_facts.columns:
        location_facts["city"] = "Unknown"

    if "circle" not in location_facts.columns:
        location_facts["circle"] = "Unknown"

    location_facts["city"] = (
        location_facts["city"].fillna("Unknown").astype(str)
    )

    location_facts["circle"] = (
        location_facts["circle"].fillna("Unknown").astype(str)
    )

else:
    print(
        "\nWARNING: Site ID could not be matched between complaints "
        "and cell_sites_df. Location fields are shown as Unknown."
    )

location_summary = (
    location_facts
    .groupby(["city", "circle"], dropna=False)
    .agg(
        complaint_count=("complaint_id", "nunique")
    )
    .reset_index()
)

assert (
    location_facts["complaint_id"].nunique()
    == complaints["complaint_id"].nunique()
), "Location merge changed the number of unique complaints."

print("\nLOCATION SUMMARY")
display(location_summary.head(10))

known_complaint_ids = set(
    complaints["complaint_id"].dropna().astype(str)
)

orphan_requests = requests[
    requests["complaint_id"].isna()
    | ~requests["complaint_id"].astype(str).isin(
        known_complaint_ids
    )
].copy()

print("\nORPHAN SERVICE REQUESTS")
print("Orphan request rows:", len(orphan_requests))
display(orphan_requests.head(10))

print("MILESTONE 07 VALIDATION")

print("Total complaints:", len(complaints))
print("Unique complaint IDs:", complaints["complaint_id"].nunique())
print("Service request summary rows:", len(requests_by_complaint))
print("Total grouped requests:", requests_by_complaint["request_count"].sum())
print("Unique customers summarized:", len(customer_repeat_count))
print("Repeat customers:", len(repeat_customers))
print("Location groups:", len(location_summary))
print("Orphan service requests:", len(orphan_requests))

assert requests_by_complaint["complaint_id"].notna().all()
assert requests_by_complaint["complaint_id"].is_unique
assert customer_repeat_count["customer_id"].is_unique
assert location_facts["complaint_id"].nunique() == complaints["complaint_id"].nunique()

print("\nMilestone 07 code executed successfully.")

SERVICE REQUEST SUMMARY
Requests with complaint references: 1500
Grouped request total: 1500


,complaint_id,request_count,open_request_count,escalated_request_count,sla_review_count
0,CMP000002,1,0,1,0
1,CMP000003,1,0,0,0
2,CMP000008,1,1,0,1
3,CMP000009,2,2,1,2
4,CMP000011,1,1,0,1
5,CMP000013,2,1,1,1
6,CMP000015,2,2,2,2
7,CMP000032,1,1,1,1
8,CMP000034,1,0,0,0
9,CMP000035,1,0,1,0



REPEAT CUSTOMER SUMMARY
Unique customers: 1217
Repeat customers: 1098


,customer_id,customer_complaint_count
0,CUST00001,5
1,CUST00002,4
2,CUST00003,6
3,CUST00004,3
4,CUST00005,10
5,CUST00006,6
6,CUST00007,6
7,CUST00008,3
8,CUST00009,2
9,CUST00010,7



LOCATION SUMMARY


,city,circle,complaint_count
0,Ahmedabad,Gujarat,349
1,Bengaluru,Karnataka,436
2,Chennai,Tamil Nadu,429
3,Delhi,Delhi NCR,436
4,Hyderabad,Andhra Pradesh & Telangana,380
5,Indore,Madhya Pradesh,448
6,Jaipur,Rajasthan,367
7,Kolkata,West Bengal,415
8,Lucknow,UP East,323
9,Mumbai,Mumbai,417



ORPHAN SERVICE REQUESTS
Orphan request rows: 3


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag,closed_request_flag,unresolved_request_flag,escalated_request_flag,sla_review_flag,_status_normalized,_is_open,_is_escalated,_sla_review
325,SR000326,CMPX943667,CUST00055,2026-05-10 10:00:00,NaT,Recharge Support,pending,Retail Ops,108.0,False,False,True,False,True,pending,1,0,1
1138,SR001139,CMPX905803,CUST01012,2026-03-31 15:00:00,NaT,Billing Investigation,,Circle Operations,86.0,True,False,True,True,True,,1,1,1
1492,SR001493,CMPX928838,CUST00696,2026-03-20 05:00:00,NaT,Network Visit,open,Fiber Field Team,NaN,False,False,True,False,True,open,1,0,1


MILESTONE 07 VALIDATION
Total complaints: 4901
Unique complaint IDs: 4901
Service request summary rows: 1265
Total grouped requests: 1500
Unique customers summarized: 1217
Repeat customers: 1098
Location groups: 13
Orphan service requests: 3

Milestone 07 code executed successfully.


## Milestone 08 · Build the complaint-level investigation view

**From the team:** CareOps needs one dependable record per original complaint with network, customer, recharge and service evidence together.

**Your responsibility:** Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.

**Handover evidence:** Create `final_cases_df` with exactly one record for every distinct complaint.

In [85]:
import pandas as pd
import numpy as np

required_objects = [
    "complaints_clean",
    "network_by_site",
    "recharge_by_complaint",
    "requests_by_complaint",
    "customer_repeat_count",
    "cell_sites_df"
]

for obj_name in required_objects:
    if obj_name not in globals():
        raise NameError(
            f"'{obj_name}' not found. Run the previous milestone "
            f"cells first, especially Milestones 1-7."
        )

complaints = complaints_clean.copy()
network_summary = network_by_site.copy()
recharge_summary = recharge_by_complaint.copy()
request_summary = requests_by_complaint.copy()
customer_summary = customer_repeat_count.copy()
sites = cell_sites_df.copy()

for df in [
    complaints,
    network_summary,
    recharge_summary,
    request_summary,
    customer_summary,
    sites
]:
    df.columns = [
        str(col).strip().lower().replace(" ", "_")
        for col in df.columns
    ]

if "complaint_id" not in complaints.columns:
    raise ValueError(
        "complaints_clean must contain complaint_id."
    )

complaints["complaint_id"] = (
    complaints["complaint_id"].astype("string").str.strip()
)

complaints = complaints[
    complaints["complaint_id"].notna()
    & complaints["complaint_id"].ne("")
].copy()

if complaints["complaint_id"].duplicated().any():
    duplicate_ids = complaints.loc[
        complaints["complaint_id"].duplicated(keep=False),
        "complaint_id"
    ].unique().tolist()

    raise ValueError(
        "complaints_clean contains duplicate complaint IDs. "
        f"Examples: {duplicate_ids[:10]}"
    )

original_complaint_ids = set(
    complaints["complaint_id"].astype(str)
)

print("Original unique complaints:", len(original_complaint_ids))

def require_columns(df, required_columns, dataframe_name):
    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )


def standardize_key(df, key):
    df = df.copy()

    if key not in df.columns:
        raise ValueError(
            f"Missing join key '{key}'. "
            f"Available columns: {df.columns.tolist()}"
        )

    df[key] = df[key].astype("string").str.strip()

    return df

if "customer_id" in complaints.columns:
    if "customer_id" in customer_summary.columns:
        complaints = standardize_key(complaints, "customer_id")
        customer_summary = standardize_key(
            customer_summary, "customer_id"
        )

        if customer_summary["customer_id"].duplicated().any():
            raise ValueError(
                "customer_repeat_count must have one row per customer."
            )

        customer_columns = [
            col for col in customer_summary.columns
            if col == "customer_id"
            or col == "customer_complaint_count"
        ]

        customer_lookup = customer_summary[
            customer_columns
        ].copy()

        complaints = complaints.merge(
            customer_lookup,
            on="customer_id",
            how="left",
            validate="many_to_one",
            suffixes=("", "_customer")
        )

    else:
        complaints["customer_complaint_count"] = 0

else:
    complaints["customer_id"] = pd.NA
    complaints["customer_complaint_count"] = 0

def find_column(df, possible_names):
    for name in possible_names:
        normalized = name.strip().lower().replace(" ", "_")

        if normalized in df.columns:
            return normalized

    return None


complaint_site_col = find_column(
    complaints,
    ["cell_site_id", "site_id", "cell_id", "site_code"]
)

site_id_col = find_column(
    sites,
    ["cell_site_id", "site_id", "cell_id", "site_code"]
)

if complaint_site_col and site_id_col:

    if complaint_site_col != "cell_site_id":
        complaints = complaints.rename(
            columns={complaint_site_col: "cell_site_id"}
        )

    if site_id_col != "cell_site_id":
        sites = sites.rename(
            columns={site_id_col: "cell_site_id"}
        )

    complaints = standardize_key(complaints, "cell_site_id")
    sites = standardize_key(sites, "cell_site_id")

    if sites["cell_site_id"].duplicated().any():
        duplicate_site_ids = sites.loc[
            sites["cell_site_id"].duplicated(keep=False),
            "cell_site_id"
        ].unique().tolist()

        print(
            "Warning: duplicate site IDs found. Keeping the first "
            "site record for each ID. Examples:",
            duplicate_site_ids[:10]
        )

        sites = sites.drop_duplicates(
            subset=["cell_site_id"],
            keep="first"
        )

    site_extra_columns = [
        col for col in sites.columns
        if col == "cell_site_id"
        or col not in complaints.columns
    ]

    site_lookup = sites[site_extra_columns].copy()

    complaints = complaints.merge(
        site_lookup,
        on="cell_site_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_site")
    )

else:
    print(
        "Warning: site ID could not be matched between complaints "
        "and cell_sites_df. Original complaints will be preserved."
    )

    if "cell_site_id" not in complaints.columns:
        complaints["cell_site_id"] = pd.NA

    if "site_status" not in complaints.columns:
        complaints["site_status"] = pd.NA

require_columns(
    network_summary,
    [
        "cell_site_id",
        "network_report_count",
        "severe_network_report_count"
    ],
    "network_by_site"
)

network_summary = standardize_key(
    network_summary, "cell_site_id"
)

if network_summary["cell_site_id"].duplicated().any():
    raise ValueError(
        "network_by_site must contain one row per cell_site_id."
    )

network_summary["network_report_count"] = pd.to_numeric(
    network_summary["network_report_count"],
    errors="coerce"
)

network_summary["severe_network_report_count"] = pd.to_numeric(
    network_summary["severe_network_report_count"],
    errors="coerce"
)

network_summary = network_summary.rename(
    columns={
        "network_report_count": "network_report_count",
        "severe_network_report_count": "severe_network_report_count"
    }
)

if "cell_site_id" in complaints.columns:

    network_columns = [
        "cell_site_id",
        "network_report_count",
        "severe_network_report_count"
    ]

    network_lookup = network_summary[network_columns].copy()

    for col in [
        "network_report_count",
        "severe_network_report_count"
    ]:
        if col in complaints.columns:
            complaints = complaints.drop(columns=[col])

    complaints = complaints.merge(
        network_lookup,
        on="cell_site_id",
        how="left",
        validate="many_to_one"
    )

else:
    complaints["network_report_count"] = 0
    complaints["severe_network_report_count"] = 0

require_columns(
    recharge_summary,
    [
        "complaint_id",
        "recharge_count",
        "recharge_risk_count"
    ],
    "recharge_by_complaint"
)

recharge_summary = standardize_key(
    recharge_summary, "complaint_id"
)

if recharge_summary["complaint_id"].duplicated().any():
    raise ValueError(
        "recharge_by_complaint must have one row per complaint_id."
    )

for col in ["recharge_count", "recharge_risk_count"]:
    recharge_summary[col] = pd.to_numeric(
        recharge_summary[col],
        errors="coerce"
    )

recharge_lookup = recharge_summary[
    [
        "complaint_id",
        "recharge_count",
        "recharge_risk_count"
    ]
].copy()

for col in ["recharge_count", "recharge_risk_count"]:
    if col in complaints.columns:
        complaints = complaints.drop(columns=[col])

complaints = complaints.merge(
    recharge_lookup,
    on="complaint_id",
    how="left",
    validate="one_to_one"
)

require_columns(
    request_summary,
    [
        "complaint_id",
        "request_count",
        "open_request_count",
        "escalated_request_count",
        "sla_review_count"
    ],
    "requests_by_complaint"
)

request_summary = standardize_key(
    request_summary, "complaint_id"
)

if request_summary["complaint_id"].duplicated().any():
    raise ValueError(
        "requests_by_complaint must have one row per complaint_id."
    )

request_metric_columns = [
    "request_count",
    "open_request_count",
    "escalated_request_count",
    "sla_review_count"
]

for col in request_metric_columns:
    request_summary[col] = pd.to_numeric(
        request_summary[col],
        errors="coerce"
    )

request_lookup = request_summary[
    ["complaint_id"] + request_metric_columns
].copy()

for col in request_metric_columns:
    if col in complaints.columns:
        complaints = complaints.drop(columns=[col])

complaints = complaints.merge(
    request_lookup,
    on="complaint_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "customer_complaint_count",
    "network_report_count",
    "severe_network_report_count",
    "recharge_count",
    "recharge_risk_count",
    "request_count",
    "open_request_count",
    "escalated_request_count",
    "sla_review_count"
]

for col in count_columns:
    if col not in complaints.columns:
        complaints[col] = 0

    complaints[col] = (
        pd.to_numeric(complaints[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

final_cases_df = complaints.copy()

assert final_cases_df["complaint_id"].notna().all(), (
    "final_cases_df contains missing complaint IDs."
)

assert final_cases_df["complaint_id"].is_unique, (
    "final_cases_df contains duplicate complaint IDs."
)

final_complaint_ids = set(
    final_cases_df["complaint_id"].astype(str)
)

assert final_complaint_ids == original_complaint_ids, (
    "Complaint IDs changed during merging. "
    f"Missing IDs: {list(original_complaint_ids - final_complaint_ids)[:10]}; "
    f"Extra IDs: {list(final_complaint_ids - original_complaint_ids)[:10]}"
)

assert len(final_cases_df) == len(original_complaint_ids), (
    "Final row count does not match the number of distinct complaints."
)

print("MILESTONE 08 — VALIDATION RESULTS")

print("Original distinct complaints:", len(original_complaint_ids))
print("Final investigation records:", len(final_cases_df))
print("Unique complaint IDs:", final_cases_df["complaint_id"].nunique())
print("Duplicate complaint IDs:", final_cases_df["complaint_id"].duplicated().sum())

print(
    "Complaint IDs preserved:",
    final_complaint_ids == original_complaint_ids
)

print("\nRequired count metrics:")
display(final_cases_df[count_columns].describe().T)

print("\nFinal investigation view — first 10 records:")
display(final_cases_df.head(10))

print("\nFinal columns:")
print(final_cases_df.columns.tolist())

print("\nMilestone 08 completed successfully.")

Original unique complaints: 4901
MILESTONE 08 — VALIDATION RESULTS
Original distinct complaints: 4901
Final investigation records: 4901
Unique complaint IDs: 4901
Duplicate complaint IDs: 0
Complaint IDs preserved: True

Required count metrics:


,count,mean,std,min,25%,50%,75%,max
customer_complaint_count,4901.0,4.990614,1.986463,1.0,4.0,5.0,6.0,11.0
network_report_count,4901.0,1.283412,1.139221,0.0,0.0,1.0,2.0,6.0
severe_network_report_count,4901.0,0.916548,0.948131,0.0,0.0,1.0,2.0,4.0
recharge_count,4901.0,0.305244,0.779240,0.0,0.0,0.0,1.0,39.0
recharge_risk_count,4901.0,0.130586,0.457238,0.0,0.0,0.0,0.0,20.0
request_count,4901.0,0.305448,0.719941,0.0,0.0,0.0,1.0,33.0
open_request_count,4901.0,0.221587,0.561949,0.0,0.0,0.0,0.0,22.0
escalated_request_count,4901.0,0.171190,0.466724,0.0,0.0,0.0,0.0,16.0
sla_review_count,4901.0,0.221587,0.561949,0.0,0.0,0.0,0.0,22.0



Final investigation view — first 10 records:


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,capacity_utilization_pct,last_maintenance_date,network_report_count,severe_network_report_count,recharge_count,recharge_risk_count,request_count,open_request_count,escalated_request_count,sla_review_count
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,<NA>,SIM activation or verification is pending afte...,...,59.2,2025-10-08,2,2,1,1,0,0,0,0
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,<NA>,SIM activation or verification is pending afte...,...,55.3,2026-03-27,2,2,0,0,1,0,1,0
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,<NA>,Fiber connection outage reported for apartment...,...,52.9,2025-10-31,1,0,1,0,1,0,0,0
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,...,84.8,2026-04-12,1,1,0,0,0,0,0,0
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,...,97.3,2026-02-19,0,0,0,0,0,0,0,0
5,CMP000006,CUST00668,SITE0682,2026-04-11,07:13,Lucknow,Up East,WhatsApp Bot,no signal,Payment was debited but recharge benefit is no...,...,57.7,2026-04-04,1,1,0,0,0,0,0,0
6,CMP000007,CUST00745,SITE0922,2026-04-03,20:19,Pune,Maharashtra,Call Center,service request pending,Fiber connection outage reported for apartment...,...,43.5,2025-07-25,1,1,1,0,0,0,0,0
7,CMP000008,CUST01112,SITE0668,2026-04-05,01:12,Kolkata,West Bengal,Call Center,internet not working,Customer is unable to use data indoors.,...,43.3,2025-10-02,1,1,0,0,1,1,0,1
8,CMP000009,CUST01060,SITE0443,2026-05-09,12:36,Ahmedabad,Gujarat,WhatsApp Bot,complaint unresolved,Payment was debited but recharge benefit is no...,...,84.8,2026-04-12,1,1,1,0,2,2,1,2
9,CMP000010,CUST01178,SITE0801,2026-02-19,14:15,Hyderabad,Andhra Pradesh & Telangana,WhatsApp Bot,slow data,Payment was debited but recharge benefit is no...,...,57.5,2025-08-29,2,2,1,0,0,0,0,0



Final columns:
['complaint_id', 'customer_id', 'cell_site_id', 'complaint_date', 'complaint_time', 'city', 'circle', 'complaint_channel', 'issue_category', 'issue_description', 'complaint_status', 'sentiment_tag', 'network_type', 'device_type', 'amount_impacted', 'invalid_complaint_date', 'issue_category_raw', 'issue_group', 'missing_customer_reference', 'missing_site_reference', 'customer_complaint_count', 'site_name', 'tower_type', 'site_status', 'avg_signal_strength_dbm', 'capacity_utilization_pct', 'last_maintenance_date', 'network_report_count', 'severe_network_report_count', 'recharge_count', 'recharge_risk_count', 'request_count', 'open_request_count', 'escalated_request_count', 'sla_review_count']

Milestone 08 completed successfully.


## Milestone 09 · Prioritize and recommend transparent next actions

**From the team:** Support leaders want explainable P0–P3 priorities based on four risk conditions, not an opaque model. They also need a draft message prompt without sending customer data externally.

**Your responsibility:** Assign P0 (3–4 signals), P1 (2), P2 (1), or P3 (0) using (1) open severe network incidents or down/inactive/outage site, (2) recharge risk, (3) escalated or SLA-review service requests, (4) three or more complaints by a customer. Produce case-specific next actions and offline AI prompt previews.

**Handover evidence:** Create four boolean risk signals, `case_priority`, `priority_score`, `recommended_action`, `ai_prompt_preview` in `final_cases_df`, and `priority_summary`.


In [86]:
import pandas as pd
import numpy as np

if "final_cases_df" not in globals():
    raise NameError(
        "final_cases_df not found. Please run Milestone 08 first."
    )

final_cases_df = final_cases_df.copy()

final_cases_df.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in final_cases_df.columns
]

if "complaint_id" not in final_cases_df.columns:
    raise ValueError(
        "final_cases_df must contain complaint_id."
    )

if final_cases_df["complaint_id"].isna().any():
    raise ValueError(
        "final_cases_df contains missing complaint IDs."
    )

if final_cases_df["complaint_id"].duplicated().any():
    raise ValueError(
        "final_cases_df must contain exactly one row per complaint."
    )

print("Input complaint records:", len(final_cases_df))

required_count_columns = [
    "severe_network_report_count",
    "recharge_risk_count",
    "escalated_request_count",
    "sla_review_count",
    "customer_complaint_count"
]

for col in required_count_columns:
    if col not in final_cases_df.columns:
        final_cases_df[col] = 0

    final_cases_df[col] = pd.to_numeric(
        final_cases_df[col],
        errors="coerce"
    ).fillna(0)

site_status_col = None

for candidate in [
    "site_status",
    "status",
    "cell_site_status",
    "network_status"
]:
    if candidate in final_cases_df.columns:
        site_status_col = candidate
        break

if site_status_col is not None:
    site_status = (
        final_cases_df[site_status_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
else:
    site_status = pd.Series(
        "",
        index=final_cases_df.index
    )

down_statuses = [
    "down",
    "inactive",
    "outage"
]

final_cases_df["severe_network_signal"] = (
    (final_cases_df["severe_network_report_count"] > 0)
    | site_status.isin(down_statuses)
).astype(bool)

final_cases_df["recharge_risk_signal"] = (
    final_cases_df["recharge_risk_count"] > 0
).astype(bool)

final_cases_df["service_followup_signal"] = (
    (final_cases_df["escalated_request_count"] > 0)
    | (final_cases_df["sla_review_count"] > 0)
).astype(bool)

final_cases_df["repeat_contact_signal"] = (
    final_cases_df["customer_complaint_count"] >= 3
).astype(bool)

risk_signal_columns = [
    "severe_network_signal",
    "recharge_risk_signal",
    "service_followup_signal",
    "repeat_contact_signal"
]

final_cases_df["priority_score"] = (
    final_cases_df[risk_signal_columns]
    .astype(int)
    .sum(axis=1)
)

def assign_priority(score):
    if score >= 3:
        return "P0"
    elif score == 2:
        return "P1"
    elif score == 1:
        return "P2"
    else:
        return "P3"


final_cases_df["case_priority"] = (
    final_cases_df["priority_score"]
    .apply(assign_priority)
)

def recommend_action(row):

    if row["severe_network_signal"]:
        return (
            "P0/Priority network investigation: verify the site status, "
            "check severe network incidents, and route the case to "
            "the network operations team. Provide a customer update "
            "after the incident is verified."
        )

    elif row["recharge_risk_signal"]:
        return (
            "Recharge investigation: verify the recharge transaction, "
            "payment status and applicable account records. Route "
            "unresolved discrepancies to the recharge support team."
        )

    elif row["service_followup_signal"]:
        return (
            "Service follow-up: review escalated requests and any "
            "available SLA-review evidence. Confirm ownership and "
            "the next follow-up step with the responsible support team."
        )

    elif row["repeat_contact_signal"]:
        return (
            "Repeat-contact review: examine the customer's complaint "
            "history, identify any unresolved issue, and coordinate "
            "a consolidated response to avoid repeated contacts."
        )

    else:
        return (
            "Routine investigation: review the complaint details, "
            "verify available evidence, and follow the standard "
            "support workflow."
        )


final_cases_df["recommended_action"] = (
    final_cases_df.apply(recommend_action, axis=1)
)

def create_ai_prompt(row):

    prompt = (
        "Draft a concise, empathetic customer-support message. "
        "This is an offline prompt preview only; do not send it "
        "to an external service.\n\n"
        f"Case priority: {row['case_priority']}\n"
        f"Risk score: {row['priority_score']} out of 4\n"
        f"Severe network signal: "
        f"{row['severe_network_signal']}\n"
        f"Recharge risk signal: "
        f"{row['recharge_risk_signal']}\n"
        f"Service follow-up signal: "
        f"{row['service_followup_signal']}\n"
        f"Repeat contact signal: "
        f"{row['repeat_contact_signal']}\n\n"
        f"Recommended internal next action: "
        f"{row['recommended_action']}\n\n"
        "Instructions: Acknowledge the reported issue, explain "
        "that the relevant team will review it, and avoid promising "
        "a resolution time unless it has been confirmed. Do not "
        "invent transaction details, technical findings, or outcomes. "
        "Use only verified information."
    )

    return prompt


final_cases_df["ai_prompt_preview"] = (
    final_cases_df.apply(create_ai_prompt, axis=1)
)


priority_order = ["P0", "P1", "P2", "P3"]

priority_summary = (
    final_cases_df["case_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("case_priority")
    .reset_index(name="case_count")
)

assert priority_summary["case_count"].sum() == len(final_cases_df), (
    "Priority summary counts do not match final_cases_df."
)

assert final_cases_df["priority_score"].between(0, 4).all(), (
    "Priority score must be between 0 and 4."
)

assert final_cases_df["case_priority"].notna().all(), (
    "Some cases have no assigned priority."
)

print("MILESTONE 09 — PRIORITY SUMMARY")

display(priority_summary)

print("\nRisk signal totals:")
risk_totals = (
    final_cases_df[risk_signal_columns]
    .sum()
    .rename("case_count")
    .reset_index()
    .rename(columns={"index": "risk_signal"})
)

display(risk_totals)

print("\nPriority distribution (%):")
priority_distribution = priority_summary.copy()

priority_distribution["percentage"] = (
    priority_distribution["case_count"]
    / max(len(final_cases_df), 1)
    * 100
).round(2)

display(priority_distribution)

print("\nP0 cases:")
p0_cases = final_cases_df[
    final_cases_df["case_priority"] == "P0"
].copy()

if len(p0_cases) > 0:
    p0_display_columns = [
        "complaint_id",
        "priority_score",
        "severe_network_signal",
        "recharge_risk_signal",
        "service_followup_signal",
        "repeat_contact_signal",
        "case_priority",
        "recommended_action"
    ]

    p0_display_columns = [
        col for col in p0_display_columns
        if col in p0_cases.columns
    ]

    display(p0_cases[p0_display_columns].head(20))
else:
    print("No P0 cases identified.")

print("\nOffline AI prompt preview examples:")

preview_columns = [
    "complaint_id",
    "case_priority",
    "ai_prompt_preview"
]

display(
    final_cases_df[preview_columns]
    .sort_values(
        "case_priority",
        key=lambda s: s.map(
            {"P0": 0, "P1": 1, "P2": 2, "P3": 3}
        )
    )
    .head(5)
)

required_output_columns = [
    "severe_network_signal",
    "recharge_risk_signal",
    "service_followup_signal",
    "repeat_contact_signal",
    "priority_score",
    "case_priority",
    "recommended_action",
    "ai_prompt_preview"
]

missing_outputs = [
    col for col in required_output_columns
    if col not in final_cases_df.columns
]

assert not missing_outputs, (
    f"Missing required outputs: {missing_outputs}"
)

print("\nTotal cases prioritized:", len(final_cases_df))
print("Priority counts reconcile:", priority_summary["case_count"].sum() == len(final_cases_df))
print("All four risk signals created:", all(
    col in final_cases_df.columns
    for col in risk_signal_columns
))
print("Offline AI prompts created:", final_cases_df["ai_prompt_preview"].notna().all())
print("\nMilestone 09 completed successfully.")


Input complaint records: 4901
MILESTONE 09 — PRIORITY SUMMARY


,case_priority,case_count
0,P0,902
1,P1,2350
2,P2,1525
3,P3,124



Risk signal totals:


,risk_signal,case_count
0,severe_network_signal,2880
1,recharge_risk_signal,580
2,service_followup_signal,1101
3,repeat_contact_signal,4438



Priority distribution (%):


,case_priority,case_count,percentage
0,P0,902,18.40
1,P1,2350,47.95
2,P2,1525,31.12
3,P3,124,2.53



P0 cases:


,complaint_id,priority_score,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_contact_signal,case_priority,recommended_action
0,CMP000001,3,True,True,False,True,P0,P0/Priority network investigation: verify the ...
1,CMP000002,3,True,False,True,True,P0,P0/Priority network investigation: verify the ...
7,CMP000008,3,True,False,True,True,P0,P0/Priority network investigation: verify the ...
8,CMP000009,3,True,False,True,True,P0,P0/Priority network investigation: verify the ...
14,CMP000015,3,True,False,True,True,P0,P0/Priority network investigation: verify the ...
21,CMP000022,3,True,True,False,True,P0,P0/Priority network investigation: verify the ...
22,CMP000023,3,True,True,False,True,P0,P0/Priority network investigation: verify the ...
23,CMP001502,4,True,True,True,True,P0,P0/Priority network investigation: verify the ...
30,CMP000032,3,True,False,True,True,P0,P0/Priority network investigation: verify the ...
40,CMP000042,3,False,True,True,True,P0,Recharge investigation: verify the recharge tr...



Offline AI prompt preview examples:


,complaint_id,case_priority,ai_prompt_preview
7,CMP000008,P0,"Draft a concise, empathetic customer-support m..."
4898,CMP004998,P0,"Draft a concise, empathetic customer-support m..."
918,CMP000946,P0,"Draft a concise, empathetic customer-support m..."
917,CMP000945,P0,"Draft a concise, empathetic customer-support m..."
915,CMP000943,P0,"Draft a concise, empathetic customer-support m..."



Total cases prioritized: 4901
Priority counts reconcile: True
All four risk signals created: True
Offline AI prompts created: True

Milestone 09 completed successfully.


## Milestone 10 · Deliver operational reports and a searchable case view

**From the team:** The CareOps supervisor needs an exportable complaint queue, circle summary and quick filtering tool that works inside Colab without external APIs.

**Your responsibility:** Sort in P0, P1, P2, P3 order. Export `jio_complaint_priority_report.csv` and `jio_careops_operations_summary.xlsx`, offer an optional widget-based search/filter with console fallback, and write 3–5 data-backed findings.

**Handover evidence:** Create `final_report`, `circle_summary`, `priority_summary`, CSV/XLSX exports and `filter_cases` helper.


In [87]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

if "final_cases_df" not in globals():
    raise NameError(
        "final_cases_df not found. Run Milestone 08 first."
    )

final_cases_df = final_cases_df.copy()

final_cases_df.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in final_cases_df.columns
]

if "complaint_id" not in final_cases_df.columns:
    raise ValueError(
        "final_cases_df must contain complaint_id."
    )

if "case_priority" not in final_cases_df.columns:
    raise ValueError(
        "case_priority is missing. Run Milestone 09 first."
    )

final_cases_df["complaint_id"] = (
    final_cases_df["complaint_id"].astype("string").str.strip()
)

final_cases_df = final_cases_df[
    final_cases_df["complaint_id"].notna()
    & final_cases_df["complaint_id"].ne("")
].copy()

if final_cases_df["complaint_id"].duplicated().any():
    raise ValueError(
        "Duplicate complaint IDs detected. "
        "Fix the duplicates before creating the report."
    )

final_cases_df["case_priority"] = (
    final_cases_df["case_priority"]
    .astype(str)
    .str.strip()
    .str.upper()
)

valid_priorities = ["P0", "P1", "P2", "P3"]

invalid_priorities = ~final_cases_df["case_priority"].isin(
    valid_priorities
)

if invalid_priorities.any():
    raise ValueError(
        "Invalid or missing priorities found. "
        "Expected P0, P1, P2, or P3."
    )

print("Input validation successful.")
print("Unique complaint records:", len(final_cases_df))

priority_order = ["P0", "P1", "P2", "P3"]

final_cases_df["case_priority"] = pd.Categorical(
    final_cases_df["case_priority"],
    categories=priority_order,
    ordered=True
)

sort_columns = ["case_priority"]

for date_col in [
    "complaint_created_date",
    "created_date",
    "complaint_date",
    "date"
]:
    if date_col in final_cases_df.columns:
        sort_columns.append(date_col)
        break

final_report = (
    final_cases_df
    .sort_values(
        by=sort_columns,
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
    .copy()
)

final_report["case_priority"] = (
    final_report["case_priority"].astype(str)
)

assert final_report["complaint_id"].is_unique
assert len(final_report) == final_report["complaint_id"].nunique()

print("\nFINAL REPORT PREVIEW")
display(final_report.head(10))

priority_summary = (
    final_report["case_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("case_priority")
    .reset_index(name="case_count")
)

assert priority_summary["case_count"].sum() == len(final_report)

print("\nPRIORITY SUMMARY")
display(priority_summary)

circle_candidates = [
    "circle",
    "circle_name",
    "region",
    "region_name",
    "telecom_circle"
]

circle_col = next(
    (
        col for col in circle_candidates
        if col in final_report.columns
    ),
    None
)

city_candidates = [
    "city",
    "city_name",
    "location",
    "location_name",
    "town"
]

city_col = next(
    (
        col for col in city_candidates
        if col in final_report.columns
    ),
    None
)

if circle_col is None:
    final_report["circle"] = "Unknown"
    circle_col = "circle"
else:
    final_report["circle"] = (
        final_report[circle_col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

customer_id_candidates = [
    "customer_id",
    "customerid",
    "customer_number"
]

customer_id_col = next(
    (
        col for col in customer_id_candidates
        if col in final_report.columns
    ),
    None
)

circle_summary = (
    final_report
    .groupby(
        ["circle", "case_priority"],
        observed=False,
        dropna=False
    )
    .agg(
        case_count=("complaint_id", "nunique")
    )
    .reset_index()
)

if customer_id_col is not None:

    customer_counts = (
        final_report
        .groupby(
            ["circle", "case_priority"],
            observed=False,
            dropna=False
        )[customer_id_col]
        .nunique()
        .reset_index(name="unique_customers")
    )

    circle_summary = circle_summary.merge(
        customer_counts,
        on=["circle", "case_priority"],
        how="left",
        validate="one_to_one"
    )

else:
    circle_summary["unique_customers"] = np.nan

circle_summary["case_priority"] = pd.Categorical(
    circle_summary["case_priority"],
    categories=priority_order,
    ordered=True
)

circle_summary = (
    circle_summary
    .sort_values(["circle", "case_priority"])
    .reset_index(drop=True)
)

circle_summary["case_priority"] = (
    circle_summary["case_priority"].astype(str)
)

print("\nCIRCLE SUMMARY")
display(circle_summary.head(20))

if "DATA_DIR" in globals():
    output_dir = os.path.abspath(str(DATA_DIR))
else:
    output_dir = "/content"

os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(
    output_dir,
    "jio_complaint_priority_report.csv"
)

excel_path = os.path.join(
    output_dir,
    "jio_careops_operations_summary.xlsx"
)

final_report.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV export completed:")
print(csv_path)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    priority_summary.to_excel(
        writer,
        sheet_name="Priority Summary",
        index=False
    )

    circle_summary.to_excel(
        writer,
        sheet_name="Circle Summary",
        index=False
    )

    final_report.to_excel(
        writer,
        sheet_name="Final Case Report",
        index=False
    )

print("\nExcel export completed:")
print(excel_path)

def filter_cases(
    priority=None,
    circle=None,
    city=None,
    issue_group=None,
    complaint_id=None,
    customer_id=None,
    search=None,
    data=None
):
    """
    Filter the complaint report by:
    - priority: P0, P1, P2, P3
    - circle
    - city
    - issue_group
    - complaint_id
    - customer_id
    - search: text search across available columns

    Returns a DataFrame and does not modify the original report.
    """

    if data is None:
        df = final_report.copy()
    else:
        df = data.copy()

    df.columns = [
        str(col).strip().lower().replace(" ", "_")
        for col in df.columns
    ]

    if priority is not None:
        if isinstance(priority, str):
            requested_priorities = [priority.upper()]
        else:
            requested_priorities = [
                str(p).upper() for p in priority
            ]

        invalid = [
            p for p in requested_priorities
            if p not in valid_priorities
        ]

        if invalid:
            raise ValueError(
                f"Invalid priority {invalid}. "
                "Use P0, P1, P2, or P3."
            )

        df = df[
            df["case_priority"].astype(str).str.upper().isin(
                requested_priorities
            )
        ]

    if circle is not None:
        circle_filter_col = next(
            (
                c for c in [
                    "circle",
                    "circle_name",
                    "region",
                    "region_name",
                    "telecom_circle"
                ]
                if c in df.columns
            ),
            None
        )

        if circle_filter_col is None:
            print("Circle column is unavailable.")
            return df.iloc[0:0].copy()

        df = df[
            df[circle_filter_col]
            .fillna("")
            .astype(str)
            .str.contains(str(circle), case=False, na=False)
        ]

    if city is not None:
        city_filter_col = next(
            (
                c for c in [
                    "city",
                    "city_name",
                    "location",
                    "location_name",
                    "town"
                ]
                if c in df.columns
            ),
            None
        )

        if city_filter_col is None:
            print("City column is unavailable.")
            return df.iloc[0:0].copy()

        df = df[
            df[city_filter_col]
            .fillna("")
            .astype(str)
            .str.contains(str(city), case=False, na=False)
        ]

    if issue_group is not None:
        issue_col = next(
            (
                c for c in [
                    "issue_group",
                    "issue_category",
                    "complaint_category",
                    "category",
                    "issue_type",
                    "complaint_type"
                ]
                if c in df.columns
            ),
            None
        )

        if issue_col is None:
            print("Issue group/category column is unavailable.")
            return df.iloc[0:0].copy()

        df = df[
            df[issue_col]
            .fillna("")
            .astype(str)
            .str.contains(
                str(issue_group),
                case=False,
                na=False
            )
        ]


    if complaint_id is not None:
        df = df[
            df["complaint_id"]
            .fillna("")
            .astype(str)
            .str.contains(
                str(complaint_id),
                case=False,
                na=False
            )
        ]

    if customer_id is not None:
        customer_col = next(
            (
                c for c in [
                    "customer_id",
                    "customerid",
                    "customer_number"
                ]
                if c in df.columns
            ),
            None
        )

        if customer_col is None:
            print("Customer ID column is unavailable.")
            return df.iloc[0:0].copy()

        df = df[
            df[customer_col]
            .fillna("")
            .astype(str)
            .str.contains(
                str(customer_id),
                case=False,
                na=False
            )
        ]

    if search is not None:
        search_mask = df.astype(str).apply(
            lambda column: column.str.contains(
                str(search),
                case=False,
                na=False,
                regex=False
            )
        ).any(axis=1)

        df = df[search_mask]

    df["_priority_sort"] = (
        df["case_priority"]
        .astype(str)
        .map({"P0": 0, "P1": 1, "P2": 2, "P3": 3})
    )

    df = (
        df.sort_values("_priority_sort")
        .drop(columns="_priority_sort")
        .reset_index(drop=True)
    )

    return df


try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    priority_widget = widgets.Dropdown(
        options=[
            ("All priorities", "ALL"),
            ("P0 — Critical", "P0"),
            ("P1 — High", "P1"),
            ("P2 — Medium", "P2"),
            ("P3 — Routine", "P3")
        ],
        value="ALL",
        description="Priority:"
    )

    circle_options = ["ALL"] + sorted(
        final_report["circle"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    circle_widget = widgets.Dropdown(
        options=circle_options,
        value="ALL",
        description="Circle:"
    )

    search_widget = widgets.Text(
        value="",
        placeholder="Complaint ID, issue, or keyword",
        description="Search:"
    )

    output_widget = widgets.Output()

    def update_case_view(change=None):
        with output_widget:
            clear_output(wait=True)

            selected_priority = (
                None if priority_widget.value == "ALL"
                else priority_widget.value
            )

            selected_circle = (
                None if circle_widget.value == "ALL"
                else circle_widget.value
            )

            results = filter_cases(
                priority=selected_priority,
                circle=selected_circle,
                search=search_widget.value or None
            )

            print("Matching cases:", len(results))
            display(results.head(100))

    priority_widget.observe(
        update_case_view,
        names="value"
    )

    circle_widget.observe(
        update_case_view,
        names="value"
    )

    search_widget.observe(
        update_case_view,
        names="value"
    )

    print("\nInteractive case filter is available.")
    display(
        widgets.VBox([
            priority_widget,
            circle_widget,
            search_widget,
            output_widget
        ])
    )

    update_case_view()

except ImportError:
    print(
        "\nipywidgets is unavailable. Use filter_cases() "
        "for console-based filtering."
    )

total_cases = len(final_report)

print("MILESTONE 10 — DATA-BACKED HANDOVER FINDINGS")

print("\n1. PRIORITY DISTRIBUTION")

for _, row in priority_summary.iterrows():
    count = int(row["case_count"])
    percentage = (
        count / total_cases * 100
        if total_cases > 0 else 0
    )

    print(
        f"{row['case_priority']}: {count} cases "
        f"({percentage:.2f}%)"
    )

urgent_count = int(
    final_report["case_priority"].isin(["P0", "P1"]).sum()
)

urgent_percentage = (
    urgent_count / total_cases * 100
    if total_cases > 0 else 0
)

print("\n2. HIGH-PRIORITY WORKLOAD")
print(
    f"{urgent_count} cases are P0/P1 "
    f"({urgent_percentage:.2f}% of all complaints)."
)

circle_volume = (
    final_report.groupby("circle")["complaint_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n3. CIRCLE WITH THE MOST COMPLAINTS")

if not circle_volume.empty:
    largest_circle = circle_volume.index[0]
    largest_circle_count = int(circle_volume.iloc[0])

    print(
        f"{largest_circle} has the highest complaint volume "
        f"with {largest_circle_count} distinct complaints."
    )
else:
    print("No circle-level records are available.")

risk_columns = [
    "severe_network_signal",
    "recharge_risk_signal",
    "service_followup_signal",
    "repeat_contact_signal"
]

available_risk_columns = [
    col for col in risk_columns
    if col in final_report.columns
]

print("\n4. MOST COMMON RISK SIGNAL")

if available_risk_columns:
    risk_counts = (
        final_report[available_risk_columns]
        .fillna(False)
        .astype(bool)
        .sum()
        .sort_values(ascending=False)
    )

    most_common_signal = risk_counts.index[0]

    print(
        f"{most_common_signal}: "
        f"{int(risk_counts.iloc[0])} cases."
    )
else:
    print("Risk-signal columns are unavailable.")

print("\n5. REPEAT-CONTACT EXPOSURE")

if "repeat_contact_signal" in final_report.columns:
    repeat_count = int(
        final_report["repeat_contact_signal"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    repeat_percentage = (
        repeat_count / total_cases * 100
        if total_cases > 0 else 0
    )

    print(
        f"{repeat_count} cases have a repeat-contact signal "
        f"({repeat_percentage:.2f}% of complaints)."
    )
else:
    print("Repeat-contact signal is unavailable.")


print("EXPORTED FILES")

for path in [csv_path, excel_path]:
    if os.path.exists(path):
        print(
            f"{os.path.basename(path)} | "
            f"{os.path.getsize(path):,} bytes | "
            f"{path}"
        )
    else:
        print("ERROR: File not found:", path)

assert os.path.exists(csv_path), "CSV export was not created."
assert os.path.exists(excel_path), "Excel export was not created."

assert final_report["complaint_id"].is_unique
assert priority_summary["case_count"].sum() == len(final_report)

print("\nFINAL REPORT ROWS:", len(final_report))
print("PRIORITY SUMMARY RECONCILES: True")
print("COMPLAINT IDS UNIQUE: True")
print("CSV EXPORT CREATED: True")
print("EXCEL EXPORT CREATED: True")
print("FILTER FUNCTION AVAILABLE:", callable(filter_cases))

print("\nMilestone 10 completed successfully.")


Input validation successful.
Unique complaint records: 4901

FINAL REPORT PREVIEW


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,escalated_request_count,sla_review_count,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_contact_signal,priority_score,case_priority,recommended_action,ai_prompt_preview
0,CMP001037,CUST00728,SITE0597,2026-02-01,19:58,Bengaluru,Karnataka,WhatsApp Bot,fiber outage,SIM activation or verification is pending afte...,...,1,1,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
1,CMP004332,CUST00471,SITE0522,2026-02-01,11:17,Chennai,Tamil Nadu,Call Center,data speed low,Customer reports unstable network for the last...,...,1,1,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
2,CMP000489,CUST01128,SITE0051,2026-02-02,11:41,Bengaluru,Karnataka,MyJio App,recharge failed,Payment was debited but recharge benefit is no...,...,0,0,True,True,False,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
3,CMP001760,CUST00353,SITE0741,2026-02-02,09:00,Pune,Maharashtra,WhatsApp Bot,no signal,SIM activation or verification is pending afte...,...,1,1,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
4,CMP002028,CUST00695,SITE0823,2026-02-02,09:28,Lucknow,Up East,Call Center,slow data,Customer says calls drop during office hours.,...,2,2,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
5,CMP002249,CUST00709,SITE0061,2026-02-02,07:40,Delhi,Delhi Ncr,Call Center,network outage,Repeated complaint from same location and serv...,...,0,0,True,True,False,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
6,CMP002568,CUST00386,SITE0888,2026-02-02,04:20,Indore,Madhya Pradesh,Call Center,<NA>,Repeated complaint from same location and serv...,...,0,0,True,True,False,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
7,CMP003842,CUST00021,SITE0120,2026-02-02,01:31,Pune,Maharashtra,Retail Store,payment deducted recharge not done,Customer says calls drop during office hours.,...,1,1,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
8,CMP003864,CUST00394,SITE0648,2026-02-02,00:39,Surat,Gujarat,Email,service request pending,Payment was debited but recharge benefit is no...,...,0,1,True,False,True,True,3,P0,P0/Priority network investigation: verify the ...,"Draft a concise, empathetic customer-support m..."
9,CMP004665,CUST00695,SITE0574,2026-02-02,05:05,Ahmedabad,Gujarat,Email,network down,Fiber connection outage reported for apartment...,...,1,1,False,True,True,True,3,P0,Recharge investigation: verify the recharge tr...,"Draft a concise, empathetic customer-support m..."



PRIORITY SUMMARY


,case_priority,case_count
0,P0,902
1,P1,2350
2,P2,1525
3,P3,124



CIRCLE SUMMARY


,circle,case_priority,case_count,unique_customers
0,Andhra Pradesh & Telangana,P0,71,71
1,Andhra Pradesh & Telangana,P1,189,177
2,Andhra Pradesh & Telangana,P2,108,105
3,Andhra Pradesh & Telangana,P3,13,13
4,Delhi Ncr,P0,88,81
5,Delhi Ncr,P1,186,171
6,Delhi Ncr,P2,156,148
7,Delhi Ncr,P3,10,10
8,Gujarat,P0,137,130
9,Gujarat,P1,374,315



CSV export completed:
/content/jio_complaint_priority_report.csv

Excel export completed:
/content/jio_careops_operations_summary.xlsx

Interactive case filter is available.


MILESTONE 10 — DATA-BACKED HANDOVER FINDINGS

1. PRIORITY DISTRIBUTION
P0: 902 cases (18.40%)
P1: 2350 cases (47.95%)
P2: 1525 cases (31.12%)
P3: 124 cases (2.53%)

2. HIGH-PRIORITY WORKLOAD
3252 cases are P0/P1 (66.35% of all complaints).

3. CIRCLE WITH THE MOST COMPLAINTS
Gujarat has the highest complaint volume with 779 distinct complaints.

4. MOST COMMON RISK SIGNAL
repeat_contact_signal: 4438 cases.

5. REPEAT-CONTACT EXPOSURE
4438 cases have a repeat-contact signal (90.55% of complaints).
EXPORTED FILES
jio_complaint_priority_report.csv | 6,330,070 bytes | /content/jio_complaint_priority_report.csv
jio_careops_operations_summary.xlsx | 1,137,200 bytes | /content/jio_careops_operations_summary.xlsx

FINAL REPORT ROWS: 4901
PRIORITY SUMMARY RECONCILES: True
COMPLAINT IDS UNIQUE: True
CSV EXPORT CREATED: True
EXCEL EXPORT CREATED: True
FILTER FUNCTION AVAILABLE: True

Milestone 10 completed successfully.


### Handover findings · 3–5 evidence-based observations
Write three to five findings supported by computed numbers, including the most affected circle/city, the actionable Critical/High workload, recharge/service risks, one operational recommendation, and at least one data-quality limitation.

**Your findings:**

-
-
-

### Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [88]:
import pandas as pd
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({
        'Milestone': task,
        'Check': criterion,
        'Result': 'Pass' if passed else 'Review',
        'Points': 5 if passed else 0,
        'Feedback': msg
    })

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(
        col in frame and pd.api.types.is_datetime64_any_dtype(frame[col])
        for col in cols
    )

def source_counts_match(summary_name, source_name, key, count_col):
    summary = df(summary_name)
    source = df(source_name)
    expected = source[key].notna().sum()
    return (
        unique_nonnull(summary, key)
        and numeric(summary, count_col)
        and summary[count_col].sum() == expected
    )

def network_counts_match():
    n = df('network_clean')
    s = df('network_by_site')
    if not has_cols('network_by_site', [
        'cell_site_id', 'network_report_count',
        'severe_network_report_count'
    ]):
        return False
    expected = (
        n.dropna(subset=['cell_site_id'])
        .groupby('cell_site_id').size()
    )
    actual = s.set_index('cell_site_id')['network_report_count']
    return (
        unique_nonnull(s, 'cell_site_id')
        and actual.reindex(expected.index).eq(expected).all()
        and actual.sum() == len(n.dropna(subset=['cell_site_id']))
    )

def issues_match():
    cases = df('final_cases_df')
    for source_name, count_col in [
        ('recharge_clean', 'recharge_count'),
        ('requests_clean', 'request_count')
    ]:
        source = df(source_name)
        if count_col not in cases.columns:
            return False
        counts = (
            source[source.complaint_id.isin(cases.complaint_id)]
            .groupby('complaint_id').size()
        )
        actual = cases.set_index('complaint_id')[count_col].fillna(0)
        expected = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(expected).all():
            return False
    return True

def flags_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    network_expected = (
        c.severe_network_report_count.fillna(0).gt(0)
        | c.site_status.astype('string').str.strip()
          .str.casefold().isin(['down', 'inactive', 'outage'])
    ).fillna(False)

    recharge_expected = c.recharge_risk_count.fillna(0).gt(0)

    service_expected = (
        c.escalated_request_count.fillna(0).gt(0)
        | c.sla_review_count.fillna(0).gt(0)
    )

    repeat_expected = c.customer_complaint_count.fillna(0).ge(3)

    return all(
        c[col].fillna(False).eq(expected).all()
        for col, expected in zip(
            flags,
            [
                network_expected,
                recharge_expected,
                service_expected,
                repeat_expected
            ]
        )
    )

def priorities_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags + [
        'priority_score', 'case_priority'
    ]):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    signals = c[flags].sum(axis=1)
    expected = signals.map(
        lambda x: 'P0' if x >= 3 else (
            'P1' if x == 2 else ('P2' if x == 1 else 'P3')
        )
    )
    actual = (
        c.case_priority.astype('string').str.strip()
        .str.extract(r'^(P[0-3])', expand=False)
    )
    return (
        c.priority_score.eq(signals).all()
        and actual.eq(expected).all()
    )

def report_matches():
    r = df('final_report')
    c = df('final_cases_df')
    return (
        has_cols('final_report', ['complaint_id', 'case_priority'])
        and len(r) == len(c)
        and unique_nonnull(r, 'complaint_id')
        and set(r.complaint_id) == set(c.complaint_id)
        and r.set_index('complaint_id').case_priority.eq(
            c.set_index('complaint_id')
            .loc[r.complaint_id, 'case_priority'].to_numpy()
        ).all()
    )

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary', [
        'case_priority', 'case_count'
    ]):
        return False
    expected = df('final_cases_df').case_priority.value_counts()
    got = s.groupby('case_priority').case_count.sum()
    return (
        got.reindex(expected.index, fill_value=0).eq(expected).all()
        and got.sum() == len(df('final_cases_df'))
    )

check(1, 'Complaint source loaded, key fields present',
      lambda: has_cols('complaints_df',
      ['complaint_id', 'customer_id', 'cell_site_id', 'complaint_status']))

check(1, 'Complaint source is non-empty',
      lambda: len(df('complaints_df')) > 0)

check(2, 'Network and recharge sources loaded',
      lambda: has_cols('network_df',
      ['network_report_id', 'cell_site_id', 'severity'])
      and has_cols('recharge_df',
      ['recharge_id', 'complaint_id', 'recharge_status'])
      and len(df('network_df')) > 0
      and len(df('recharge_df')) > 0)

check(2, 'Service request source loaded',
      lambda: has_cols('requests_df',
      ['service_request_id', 'complaint_id', 'service_status'])
      and len(df('requests_df')) > 0)

check(3, 'One clean record per complaint',
      lambda: unique_nonnull(df('complaints_clean'), 'complaint_id')
      and len(df('complaints_clean')) <= len(df('complaints_df')))

check(3, 'Complaint dates and categories standardized',
      lambda: real_dates(df('complaints_clean'), ['complaint_date'])
      and has_cols('complaints_clean',
      ['issue_group', 'complaint_status', 'city', 'circle']))

check(4, 'Operational records cleaned',
      lambda: unique_nonnull(df('network_clean'), 'network_report_id')
      and unique_nonnull(df('recharge_clean'), 'recharge_id')
      and unique_nonnull(df('requests_clean'), 'service_request_id'))

check(4, 'Operational dates standardized',
      lambda: real_dates(df('network_clean'), ['report_date'])
      and real_dates(df('recharge_clean'), ['recharge_date'])
      and real_dates(df('requests_clean'), ['request_created_date']))

check(5, 'Network reports aggregated by site',
      lambda: unique_nonnull(df('network_by_site'), 'cell_site_id')
      and has_cols('network_by_site',
      ['network_report_count', 'severe_network_report_count']))

check(5, 'Network report totals reconcile',
      network_counts_match)

check(6, 'Recharge summary uses unique complaint IDs',
      lambda: unique_nonnull(df('recharge_by_complaint'), 'complaint_id')
      and has_cols('recharge_by_complaint',
      ['recharge_count', 'recharge_risk_count']))

check(6, 'Recharge totals match cleaned source',
      lambda: source_counts_match(
          'recharge_by_complaint', 'recharge_clean',
          'complaint_id', 'recharge_count'
      ))

check(7, 'Service requests summarized by complaint',
      lambda: unique_nonnull(df('requests_by_complaint'), 'complaint_id')
      and has_cols('requests_by_complaint',
      ['request_count', 'open_request_count',
       'escalated_request_count', 'sla_review_count'])
      and source_counts_match(
          'requests_by_complaint', 'requests_clean',
          'complaint_id', 'request_count'
      ))

check(7, 'Repeat customer counts reconcile',
      lambda: has_cols('customer_repeat_count',
      ['customer_id', 'customer_complaint_count'])
      and unique_nonnull(df('customer_repeat_count'), 'customer_id')
      and df('customer_repeat_count').customer_complaint_count.sum()
      == df('complaints_clean').customer_id.notna().sum())

check(8, 'Master table contains all complaints once',
      lambda: unique_nonnull(df('final_cases_df'), 'complaint_id')
      and set(df('final_cases_df').complaint_id)
      == set(df('complaints_clean').complaint_id))

check(8, 'Master issue counts reconcile',
      issues_match)

check(9, 'Operational risk signals follow business rules',
      flags_match)

check(9, 'Priority mapping matches risk signals',
      priorities_match)

check(10, 'Detailed report and export exist',
      lambda: report_matches()
      and (Path(DATA_DIR) / 'jio_complaint_priority_report.csv').is_file())

check(10, 'Priority summary and export exist',
      lambda: summary_matches()
      and (Path(DATA_DIR) / 'jio_careops_operations_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)

score_by_task = assessment_df.groupby(
    'Milestone', as_index=False
).agg(
    Points=('Points', 'sum'),
    Passed=('Result', lambda x: (x == 'Pass').sum())
)

score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({
    0: 'Review',
    1: 'Partially verified',
    2: 'Checks passed'
})

print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone', 'Points', 'Maximum', 'Status']])

print(f'Automated checks score: {assessment_df.Points.sum()}/100')

print('Checks needing attention:')
display(
    assessment_df.loc[
        assessment_df.Result == 'Review',
        ['Milestone', 'Check', 'Feedback']
    ].reset_index(drop=True)
)

import json as _grade_json

print(
    'JIO_CAREOPS_ASSESSMENT_JSON='
    + _grade_json.dumps({
        'points': int(assessment_df.Points.sum()),
        'milestones': score_by_task[
            ['Milestone', 'Points', 'Maximum', 'Status']
        ].to_dict('records'),
        'checks': assessment_df[
            ['Milestone', 'Check', 'Result', 'Points']
        ].to_dict('records')
    })
)


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


JIO_CAREOPS_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Complaint source loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Complaint source is non-empty", "Result": "Pass", "Point

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
